# 10 · Goal-driven simulation: can a simulated hand reproduce the real EMG, and what does its controller learn?

**Investigation ladder** (see [`EVALUATION.md`](../EVALUATION.md)):
- **Stage 8 · Goal-driven simulation: does a simulated hand + controller reproduce the structure of the real EMG, and which ingredient causes it?**

**Read first:** Almani, Lazzari, Walker & Saxena 2026, *Embodied sensorimotor control* (Annu Rev Biomed Eng), especially section 3.5 and the sidebar on neural constraints; Lazzari & Saxena 2025, *Multitasking recurrent networks utilize compositional strategies for control of movement* (bioRxiv 10.1101/2025.09.10.675375); Sussillo & Barak 2013 (fixed points); Kutch & Valero-Cuevas 2012 (can biomechanics alone produce 'synergies'?)

**We expect:** a controller trained only to **imitate the recorded hand movements** through a simulated hand produces EMG that resembles the real EMG in *structure*: a few synergies, a small pose-relevant subspace that is not the synergies, and EMG more shared across tasks than pose. Shared tendons and electrode crosstalk explain part of this. A controller that must also match the real EMG fits it better, but needs more effort and co-contraction than the body calls efficient. **Wrong if:** the simulated EMG reproduces none of the real signatures, the ablations change nothing, or the two controllers are indistinguishable.

**The idea.** Every earlier notebook fitted models to *recorded* signals. Here we flip it around, as Almani et al.
describe for embodied control and Lazzari & Saxena do for movement: build a **body**, give it a **controller**, give the
controller a **goal**, and see what muscle activity *emerges*. Our goal is to imitate this person's recorded hand
movements. The controller only ever sees **pose**. Its muscle activity is then turned into simulated electrode signals
and compared with the real EMG it never saw.

**Two conditions, trained the same way (same body, same network, same budget, same seeds):**

| | Condition 1 · goal-driven | Condition 2 · EMG-constrained |
|---|---|---|
| Loss | pose error + effort + regularisers | the same **+ λ × mismatch with the real EMG** (Almani et al.'s "data-driven" constraint) |
| Uses the EMG for training? | **No** | Yes (training frames of training tasks only) |
| What it can test | **emergence**: does body + task produce the real EMG structure? | a model of **this person's** control: what muscle activity explains both their pose and their EMG? |
| What it cannot test | the nervous system's actual solution | emergence (it was trained on the EMG, so matching it proves nothing) |

**Honesty rules, because this is one person's data (not the full dataset):**
1. Every body and electrode parameter is **fixed in advance** from rough literature values (listed in `ref10.py`).
   None is fitted to this person's EMG; that would make any match circular. Only the rest posture and joint limits
   come from this person's **pose**.
2. Condition 2 gets **no free per-channel gains or crosstalk widths**. The only EMG-dependent choices are the band's
   rotation/flip (one of 32 discrete options per hand, picked on training frames) and λ (picked on a training-only
   validation slice).
3. Everything is within-person: the independent units are the 15 recording blocks (`protocol`), and every claim is
   checked on **held-out time** and on **held-out tasks** the controller never saw.
4. "Virtual people" (other band positions, crosstalk, tendon coupling) at the end are **hypotheses for stage 7**,
   not evidence about other people.

**How this notebook works** (the same in every notebook)
- **`# TODO`** cells are yours. Each has a docstring with the input and output shapes. You write the core
  algorithms yourself in NumPy.
- Under every TODO there are three **💡 hints** that open one at a time: the plan in words, the tools with a tiny
  example, then a skeleton with blanks (`____`).
- After each TODO a **✅ check cell** compares your answer with the reference in [`ref10.py`](ref10.py), then a **📊 plot**
  shows what your code did. Other notebooks use the reference, so your blanks never break theirs.
- **✍️ questions** are for your own words, in the markdown cell under them.
- The **Result** cell at the end is the **gate**: it decides whether the next stage of the investigation is worth running.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** ➕ beyond the proposal: stage 8 of the ladder. It asks *why* the real EMG has the structure stages 1–6 found, and its transfer test is another within-person pilot of H1.

**Goal:** build a differentiable hand, train controllers to imitate the recorded movements with and without the EMG, and use them to explain the real EMG's structure and to reverse-engineer what a controller learns.

**Why:** if body mechanics and the recording alone produce 'synergies', a low-variance pose subspace and EMG that is more shared than pose, then those findings say more about the hand and the electrodes than about the nervous system (Kutch & Valero-Cuevas 2012).

**Milestones**
1. Body pieces written from scratch and checked: activation dynamics, tendon-coupled torques, integration step, electrode ring
2. Controller pieces and both losses written from scratch; checkpoints for 2 conditions × 3 seeds × 2 folds loaded
3. Imitation quality and EMG prediction on held-out time and held-out tasks, with CIs over blocks
4. Real-EMG signatures reproduced (or not) by each condition; the tendon × crosstalk ablation
5. Reverse-engineering: fixed points, linear approximation, task-vector geometry, unit clusters, readout, lesions, robustness across seeds
6. Compositional transfer and virtual people

**Gate (decides the next stage):** the goal-driven simulation reproduces **at least 2** of the 3 real-EMG signatures (synergy count, low-variance pose subspace that is not the synergies, EMG more shared than pose) **and** the ablation identifies which ingredient (tendon coupling, crosstalk) is necessary for each; interpretation results count only if they hold across all 3 seeds.

**Done when:** every ✅ check passes and the gate table is filled in with numbers that hold across seeds.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

_d = ROOT / "notebooks" / "experiments" / "10_goal_driven_simulation"
assert (_d / "ref10.py").exists(), f"{_d}/ref10.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref10", None)
_d = ROOT / "notebooks" / "experiments" / "02_how_many_synergies"
assert (_d / "ref02.py").exists(), f"{_d}/ref02.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref02", None)
_d = ROOT / "notebooks" / "experiments" / "06_shared_subspace"
assert (_d / "ref06.py").exists(), f"{_d}/ref06.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref06", None)
_d = ROOT / "notebooks" / "experiments" / "04_compositional_structure"
assert (_d / "ref04.py").exists(), f"{_d}/ref04.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref04", None)
_d = ROOT / "notebooks" / "experiments" / "07_dynamics"
assert (_d / "ref07a.py").exists(), f"{_d}/ref07a.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref07a", None)
_d = ROOT / "notebooks" / "experiments" / "03_are_synergies_real"
assert (_d / "ref03.py").exists(), f"{_d}/ref03.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref03", None)
_d = ROOT / "notebooks" / "experiments" / "common"
sys.path.insert(0, str(_d)); sys.modules.pop("protocol", None)
import protocol                 # shared evaluation protocol: sessions, folds, baselines, nulls, CIs
import ref10 as ref          # the reference answers, used only by the ✅ check cells

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01's prep.py (data/ is never committed)

In [ ]:
# ── Load the shared data (rebuilds it with prep.py if missing or built by an older version) ──
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(ROOT / "notebooks" / "experiments" / "01_data_prep")); sys.modules.pop("prep", None)
import prep
if not prep.is_current(PROCESSED):
    subprocess.run([sys.executable, "notebooks/experiments/01_data_prep/prep.py"], check=True)
d = np.load(PROCESSED)
import json
import torch
import ref02, ref03, ref04, ref06, ref07a

torch.set_num_threads(min(8, os.cpu_count() or 1))
NB10 = ROOT / "notebooks" / "experiments" / "10_goal_driven_simulation"
CKPT = NB10 / "checkpoints"
SUMMARY = json.loads((CKPT / "training_summary.json").read_text())

emg        = d["emg"].astype(np.float64)    # (N, 16) real EMG envelope (NEVER used to train condition 1)
pose       = d["pose"].astype(np.float64)   # (N, 20) joint angles in degrees (data version 2)
pose_valid = d["pose_valid"]                # (N,)
rec, split = d["rec"], d["split"]           # (N,)
stage_of_rec, side_of_rec, filename_of_rec = d["stage"], d["side"], d["filename"]
train, test = split == 0, split == 1

block  = protocol.block_of_rec(filename_of_rec)[rec]              # (N,) the independent unit (15 blocks)
groups = protocol.stage_groups(stage_of_rec, filename_of_rec)    # 14 task groups
stages = list(np.unique(stage_of_rec))                           # 15 stage names = the task inputs
task_of_rec = np.array([stages.index(s) for s in stage_of_rec])  # (30,) task index of each recording
n_tasks = len(stages)
task = task_of_rec[rec]                                          # (N,) task of every frame
target = ref.ffill_targets(pose, pose_valid, rec)                # (N, 20) imitation targets (dropouts filled)
fold_info = {fold: ref.fold_of_groups(groups, fold) for fold in (0, 1)}   # (held-out names, held-out recs, train recs)
ROT_F = {f: tuple(SUMMARY["rotation_by_fold"][str(f)]) for f in (0, 1)}  # band (rotation, flip) per fold: left, right
LAM_F = {f: SUMMARY["lam_emg_by_fold"][str(f)] for f in (0, 1)}          # condition-2 weight per fold
JOINT_NAMES, MUSCLES = ref.JOINT_NAMES, ref.MUSCLES
SEEDS = [0, 1, 2]
print(f"{len(emg)} frames | {n_tasks} tasks | {len(groups)} task groups | {len(np.unique(block))} blocks | "
      f"{len(MUSCLES)} lumped muscles | band orientation per fold {ROT_F} | λ_emg per fold {LAM_F}")

### The variables you start with
| Name | Shape | What it is |
|---|---|---|
| `emg`, `pose`, `pose_valid`, `rec`, `split`, `train`, `test` | (N, …) | as in every notebook |
| `block`, `groups` | (N,), 14 | independent units (15 blocks) and task groups (protocol) |
| `stages`, `task_of_rec`, `task`, `n_tasks` | 15 | the task inputs of the controller: one per stage name |
| `target` | (N, 20) | what the controller must imitate (pose with tracking dropouts filled in) |
| `fold_info` | 2 | per fold: (held-out task groups, their recordings, the fold's training recordings) |
| `ROT_F`, `LAM_F` | 2 | per fold: the band's (rotation, flip) for left and right recordings, and λ_emg, both chosen on that fold's **training** recordings only |
| `JOINT_NAMES`, `MUSCLES` | 20, 32 | joints and lumped muscles of the simulated hand |

Letters in docstrings: **B** batch, **T** frames, **J** = 20 joints, **M** = 32 muscles, **H** = 128 hidden units,
**E** = 16 electrodes, **S** = 15 tasks.

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

---
## Part 1 · Build the body
The simulated hand has the dataset's **20 joints**, driven by **32 lumped muscles**:

| Group | Muscles | Where | Seen by the wristband? |
|---|---|---|---|
| Long finger flexors | FDP and FDS, one slip per finger | forearm, volar side (FDP deep) | yes |
| Long finger extensors | EDC slips, EIP (index), EDM (pinky) | forearm, dorsal side | yes |
| Long thumb muscles | FPL, EPL, EPB, APL | forearm | yes |
| Hand muscles | thenar, adductor pollicis, a +/− sideways pair per finger (interossei), lumbricals | inside the hand | barely |

Four pieces turn muscle commands into movement and electrode signals. You write each one; after the checks, the rest
of the notebook uses the reference versions so everybody's simulation is identical.

### Task 1 · Activation dynamics
A muscle can't switch on instantly. Its activation `a` follows the neural excitation `u` with a time constant: about
**10 ms when rising** and **40 ms when falling** (Zajac 1989). The exact update for one step of length `dt`:

$$a \leftarrow a + \left(1 - e^{-dt/\tau}\right)(u - a), \qquad \tau = \begin{cases}\tau_{act} & u > a\\ \tau_{deact} & \text{otherwise}\end{cases}$$

This form never overshoots, whatever `dt` is.

In [ ]:
def activation_step(a: torch.Tensor, u: torch.Tensor, dt: float) -> torch.Tensor:
    """One step of first-order activation dynamics.

    Parameters
    ----------
    a  : (B, M) current activations in [0, 1]
    u  : (B, M) excitations in [0, 1]
    dt : step length in seconds

    Returns
    -------
    (B, M) new activations. Time constant ref.TAU_ACT where u > a, ref.TAU_DEACT elsewhere.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Build a tensor of time constants the same shape as `a`: `ref.TAU_ACT` where `u > a`, `ref.TAU_DEACT` elsewhere.
2. The fraction of the gap closed this step is `1 − exp(−dt / τ)`.
3. Move `a` that fraction of the way towards `u`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `torch.where(cond, x, y)` picks element by element; `torch.full_like(a, value)` makes a tensor of one value.
- `torch.exp` works element by element.

```python
a = torch.tensor([0.0, 1.0]); u = torch.tensor([1.0, 0.0])
torch.where(u > a, torch.full_like(a, 0.01), torch.full_like(a, 0.04))   # tensor([0.0100, 0.0400])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def activation_step(a, u, dt):
    tau = torch.where(u > a, torch.full_like(a, ref.TAU_ACT), torch.full_like(a, ____))
    return a + (1.0 - torch.exp(-dt / ____)) * (u - ____)
```

</details>

In [ ]:
# ✅ Check Task 1
g = torch.Generator().manual_seed(0)
a0, u0 = torch.rand(4, ref.M, generator=g), torch.rand(4, ref.M, generator=g)
assert torch.allclose(activation_step(a0, u0, 0.005), ref.activation_step(a0, u0, 0.005), atol=1e-6), \
    "activation differs: check which time constant goes with rising (u > a) and falling"
assert torch.allclose(activation_step(a0, u0, 10.0), u0, atol=1e-4), "a very long step should reach u exactly"
print("✅ activation dynamics work")

# 📊 What to look for: the rise (10 ms) is faster than the fall (40 ms)
a_t, out = torch.zeros(1, 1), []
for k in range(80):
    a_t = activation_step(a_t, torch.ones(1, 1) if k < 30 else torch.zeros(1, 1), 0.005); out.append(float(a_t))
plt.figure(figsize=(6, 2.5)); plt.plot(np.arange(80) * 5, out); plt.axvline(150, color="gray", ls=":")
plt.xlabel("time (ms)"); plt.ylabel("activation"); plt.title("excitation on for 150 ms, then off"); plt.show()

✍️ **Q1.** Why would a slow *deactivation* matter for the EMG–pose lag we found in notebook 06 (EMG leads pose by about 200 ms)? Is 40 ms enough to explain it, and if not, what else in the body adds delay?

> *Your answer:*

### Task 2 · Muscle forces → joint torques, with shared tendons
Each muscle pulls with force `F_max × a`. A **moment arm** turns that force into a torque at every joint it crosses
(`ref.base_moment_arms()`, a 32 × 20 matrix: + flexes / + moves sideways, − extends). The finger slips of FDP, FDS and
EDC are not independent: FDP's slips share a muscle belly and EDC's are tied together by bands between the tendons
(juncturae). We model this with a **coupling matrix** `C(c)` (4 × 4 over index, middle, ring, pinky): a share `1 − c`
of a slip's force stays on its own finger and `c` is split between its neighbours. Rows sum to 1.

`ref.coupled_moment_arms(R, c_scale)` uses your `coupling_matrix` idea to spread each slip's moment arms over the
fingers (c = 0.3 for FDP and EDC, 0.1 for FDS, all multiplied by `c_scale`; `c_scale = 0` cuts the coupling).

In [ ]:
def coupling_matrix(c: float) -> np.ndarray:
    """(4, 4) share of finger slip i's force reaching finger j (order: index, middle, ring, pinky).

    Diagonal = 1 − c. The rest, c, is split equally between the slip's neighbours: index and pinky have one
    neighbour (it gets all of c), middle and ring have two (c / 2 each). Every row sums to 1.
    """
    # TODO
    raise NotImplementedError


def muscle_torques(a: torch.Tensor, R_eff: torch.Tensor, F: torch.Tensor) -> torch.Tensor:
    """Joint torques from activations.

    a     : (B, M) activations
    R_eff : (M, J) moment arms (with tendon coupling)
    F     : (M,) relative maximum force of each muscle

    Returns (B, J): ref.TORQUE_SCALE * (force of each muscle) @ R_eff.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `coupling_matrix`: start from zeros; for each finger i put `1 − c` on the diagonal, list its neighbours
   (i − 1 and i + 1 if they exist), and give each neighbour `c / number_of_neighbours`.
2. `muscle_torques`: force = `a * F` (broadcast over the batch), then a matrix product with `R_eff`, times the
   torque scale.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `[j for j in (i - 1, i + 1) if 0 <= j < 4]` lists the neighbours.
- `(B, M) @ (M, J)` → `(B, J)`.

```python
F = torch.tensor([1.0, 2.0]); a = torch.tensor([[0.5, 0.5]])
a * F                                            # tensor([[0.5, 1.0]])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def coupling_matrix(c):
    C = np.zeros((4, 4))
    for i in range(4):
        nb = [j for j in (i - 1, i + 1) if 0 <= j < 4]
        C[i, i] = ____
        for j in nb:
            C[i, j] = c / len(____)
    return C


def muscle_torques(a, R_eff, F):
    return ref.TORQUE_SCALE * (a * ____) @ ____
```

</details>

In [ ]:
# ✅ Check Task 2
for c in (0.0, 0.3, 1.0):
    assert np.allclose(coupling_matrix(c), ref.coupling_matrix(c)), f"coupling_matrix({c}) differs"
    assert np.allclose(coupling_matrix(c).sum(1), 1.0), "rows must sum to 1"
R_eff = torch.tensor(ref.coupled_moment_arms(ref.base_moment_arms(), 1.0), dtype=torch.float32)
F = torch.tensor(ref.strengths(), dtype=torch.float32)
assert torch.allclose(muscle_torques(a0, R_eff, F), ref.muscle_torques(a0, R_eff, F), atol=1e-3), \
    "muscle_torques differs: force = a * F (element by element), then @ R_eff, then times ref.TORQUE_SCALE"
print("✅ tendon coupling and torques work")

# 📊 What to look for: with coupling, one slip of FDP also pulls on the neighbouring fingers
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
for k, cs in enumerate((0.0, 1.0)):
    Rk = ref.coupled_moment_arms(ref.base_moment_arms(), cs)
    im = ax[k].imshow(Rk[:18], cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax[k].set_yticks(range(18), MUSCLES[:18], fontsize=6); ax[k].set_xticks(range(20), JOINT_NAMES, rotation=90, fontsize=6)
    ax[k].set_title(f"forearm muscles' moment arms, c_scale = {cs}")
plt.colorbar(im, ax=ax); plt.show()

✍️ **Q2.** Look at the FDP_MIDDLE row with and without coupling. Notebook 06 found that a 'shared finger latent' did **not** beat its null because the fingers' poses are correlated. Would tendon coupling make the *EMG* of the four fingers more alike, the *pose*, or both? Explain through the chain: command → muscle → tendon → joint.

> *Your answer:*

### Task 3 · One integration step
Each joint is a rotating mass driven by four torques: muscles, a **passive spring** pulling towards the rest posture
(the person's median training pose), **damping**, and a stiff **soft limit** outside the observed range (1st–99th
percentile of training pose). Each fold gets its **own body**, built only from the training split of the recordings that
fold trains on (`ref.fold_body`): a task the controller is tested on as "unseen" must not shape the hand it moves. Per joint:

$$I\,\ddot q = \tau_{muscle} - k\,(q - q_{rest}) - b\,\dot q - k_{lim}\,\big(\mathrm{relu}(q - q_{hi}) - \mathrm{relu}(q_{lo} - q)\big)$$

**Semi-implicit Euler** updates the velocity first and then moves the position with the **new** velocity. It is as
cheap as plain Euler but much more stable for springs. One 20 ms frame = `ref.SUBSTEPS` = 4 steps of 5 ms.

In [ ]:
def body_step(q: torch.Tensor, qd: torch.Tensor, tau: torch.Tensor, p: "ref.BodyParams", dt: float
              ) -> tuple[torch.Tensor, torch.Tensor]:
    """One semi-implicit Euler step.

    q, qd : (B, J) joint angles (radians) and velocities (rad/s)
    tau   : (B, J) muscle torques
    p     : body parameters: p.q_rest, p.q_lo, p.q_hi, p.I, p.k, p.b (each (J,)); ref.LIMIT_STIFFNESS
    dt    : step (s)

    Returns new (q, qd): acceleration from the equation above, then qd += dt * acc, then q += dt * (new qd).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. How far outside the limits is each joint? `over = relu(q − q_hi) − relu(q_lo − q)` (positive above, negative below).
2. Acceleration = (muscle torque − spring − damping − limit term) / inertia.
3. Update the velocity, then the position with that new velocity.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `torch.relu(x)` = max(x, 0) element by element.
- Every body parameter is a (J,) tensor, so it broadcasts over the batch.

```python
q = torch.tensor([[0.0, 2.0]]); hi = torch.tensor([1.0, 1.0])
torch.relu(q - hi)        # tensor([[0., 1.]]): only the second joint is over its limit
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def body_step(q, qd, tau, p, dt):
    over = torch.relu(q - p.q_hi) - torch.relu(p.q_lo - q)
    acc = (tau - p.k * (q - p.q_rest) - p.b * qd - ref.LIMIT_STIFFNESS * ____) / p.I
    qd = qd + dt * ____
    q = q + dt * ____
    return q, qd
```

</details>

In [ ]:
# ✅ Check Task 3
# One body PER FOLD: rest posture and joint limits from the training split of that fold's TRAINING recordings only,
# so the tasks a fold holds out never shape its body.
body_f = {f: ref.fold_body(pose, pose_valid, rec, split, fold_info[f][2]) for f in (0, 1)}
body_c0_f = {f: ref.fold_body(pose, pose_valid, rec, split, fold_info[f][2], c_scale=0.0) for f in (0, 1)}  # tendons cut
body, body_c0 = body_f[0], body_c0_f[0]                       # fold 0's bodies for the checks and plots below
q0 = body.q_rest[None] + 0.3 * torch.randn(3, ref.J, generator=g)
qd0, tau0 = torch.randn(3, ref.J, generator=g), 100 * torch.randn(3, ref.J, generator=g)
mine, theirs = body_step(q0, qd0, tau0, body, 0.005), ref.body_step(q0, qd0, tau0, body, 0.005)
assert all(torch.allclose(x, y, atol=1e-5) for x, y in zip(mine, theirs)), "body_step differs from the reference"
q_, qd_, a_ = q0.clone(), torch.zeros_like(q0), torch.zeros(3, ref.M)
for _ in range(250):                                           # 5 s with no muscle input
    q_, qd_, a_ = ref.frame_step(q_, qd_, a_, torch.zeros(3, ref.M), body)
assert float((q_ - body.q_rest).abs().max()) < 1e-2, "with no input the hand should settle at its rest posture"
print("✅ integration step works; with no input the hand settles at rest")

# 📊 What to look for: one FDP slip at half excitation, with and without tendon coupling
fig, ax = plt.subplots(1, 2, figsize=(13, 3), sharey=True)
for k, (bd, name) in enumerate(((body, "coupled"), (body_c0, "uncoupled"))):
    qq, qv, aa = bd.q_rest[None].clone(), torch.zeros(1, ref.J), torch.zeros(1, ref.M)
    u = torch.zeros(1, ref.M); u[0, MUSCLES.index("FDP_MIDDLE")] = 0.5
    for _ in range(25):
        qq, qv, aa = ref.frame_step(qq, qv, aa, u, bd)
    ax[k].bar(range(20), torch.rad2deg(qq - bd.q_rest)[0].numpy()); ax[k].set_title(f"FDP_MIDDLE for 0.5 s, {name}")
    ax[k].set_xticks(range(20), JOINT_NAMES, rotation=90, fontsize=6)
ax[0].set_ylabel("change in angle (°)"); plt.tight_layout(); plt.show()

✍️ **Q3.** In the uncoupled hand, which joints move when only FDP_MIDDLE is active, and why are there still small movements elsewhere (hint: what else connects the joints in this model, and what doesn't)?

> *Your answer:*

### Task 4 · The electrode ring
16 electrodes sit evenly around the forearm. A forearm muscle at angle θ<sub>m</sub> reaches electrode *e* at
θ<sub>e</sub> with a **von Mises** fall-off:

$$G_{em} = g_m \exp\!\big(\kappa_m(\cos(\theta_e - \theta_m) - 1)\big)$$

`g_m` is lower for deep muscles (FDP, FPL, EIP, EPL); **κ (kappa) sets crosstalk**: large κ = narrow pick-up, small κ
= one muscle reaches many electrodes. Hand muscles are far from the wrist, so they reach every electrode with a small
constant gain. `ref.muscle_positions()` gives θ, g and κ for every muscle (κ = 0 marks a hand muscle). The band's
unknown placement is a rotation of the rows and, for a mirrored band, a reversal.

In [ ]:
def electrode_gains(n_electrodes: int = 16, kappa_scale: float = 1.0, rotation: int = 0, flip: bool = False
                    ) -> np.ndarray:
    """(n_electrodes, M) mixing matrix from muscle activations to electrode signals.

    Electrode e sits at angle 2π e / n_electrodes. For a forearm muscle (kappa > 0):
    gain * exp(kappa_scale * kappa * (cos(theta_e − theta_m) − 1)); for a hand muscle (kappa == 0): its gain at every
    electrode. Then, if flip: reverse the row order; finally roll the rows by `rotation` (np.roll, axis 0).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 4? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Get `ang, gain, kap = ref.muscle_positions()` and the electrode angles.
2. Fill the matrix column by column (one column per muscle): a constant for hand muscles, the von Mises profile for
   forearm muscles.
3. Apply the flip, then the rotation.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `theta = 2 * np.pi * np.arange(n) / n` → electrode angles.
- `G[::-1]` reverses the rows; `np.roll(G, r, axis=0)` shifts them by r.

```python
np.exp(3.0 * (np.cos(np.array([0.0, np.pi])) - 1))   # [1.0, 0.0025]: next to the muscle vs opposite side
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def electrode_gains(n_electrodes=16, kappa_scale=1.0, rotation=0, flip=False):
    ang, gain, kap = ref.muscle_positions()
    theta = 2 * np.pi * np.arange(n_electrodes) / n_electrodes
    G = np.zeros((n_electrodes, ref.M))
    for m in range(ref.M):
        if kap[m] == 0.0:
            G[:, m] = ____
        else:
            G[:, m] = gain[m] * np.exp(kappa_scale * kap[m] * (np.cos(theta - ____) - 1.0))
    if flip:
        G = G[::-1]
    return np.roll(G, ____, axis=0)
```

</details>

In [ ]:
# ✅ Check Task 4
for kw in ({}, {"kappa_scale": 4.0}, {"rotation": 5, "flip": True}):
    assert np.allclose(electrode_gains(**kw), ref.electrode_gains(**kw)), f"electrode_gains({kw}) differs"
print("✅ electrode ring works")

# 📊 What to look for: wide pick-up (crosstalk) vs narrow; deep muscles are weaker; hand muscles barely register
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5), sharey=True)
for k, ks in enumerate((1.0, 4.0)):
    ax[k].imshow(electrode_gains(kappa_scale=ks), aspect="auto", cmap="magma")
    ax[k].set_xticks(range(ref.M), MUSCLES, rotation=90, fontsize=6); ax[k].set_title(f"kappa_scale = {ks}")
ax[0].set_ylabel("electrode"); plt.tight_layout(); plt.show()

✍️ **Q4.** With wide pick-up (kappa_scale = 1), how many electrodes 'hear' FDS_MIDDLE above a third of its peak? Predict what crosstalk does to the number of synergies NMF will need for the simulated EMG, and why.

> *Your answer:*

---
## Part 2 · The controller and two ways to train it
The controller is a recurrent network in the form Lazzari & Saxena use: a leaky state `h` (H = 128 units) updated
every 20 ms frame

$$h \leftarrow h + \alpha\big(-h + W_h \tanh(h) + W_u x + b\big), \qquad u = \sigma\big(W_o \tanh(h) + b_o\big)$$

with α = 0.5 and `u` the 32 muscle excitations (σ = sigmoid keeps them in 0–1). Its input `x` each frame:
- the **task cue**: a one-hot vector over the 15 stages (which movement this is);
- the **reference**: the target pose 100 ms ahead, reduced to its **top 3 eigenpostures** (so the network must
  fill in the rest from the task and from the body's feedback);
- **proprioception**: the body's joint angles and velocities from 40 ms ago (sensory delay).

The network never sees the EMG as an input.

In [ ]:
def rnn_step(h: torch.Tensor, x: torch.Tensor, W_h: torch.Tensor, W_u: torch.Tensor, b: torch.Tensor,
             alpha: float) -> torch.Tensor:
    """One leaky recurrent update. Rows are batch items.

    h : (B, H), x : (B, n_in), W_h : (H, H), W_u : (H, n_in), b : (H,)
    Returns h + alpha * (−h + tanh(h) @ W_h' + x @ W_u' + b).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 5? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

With rows as batch items, "W_h times tanh(h)" becomes `tanh(h) @ W_h.T`. Same for the input. Add the bias, subtract h,
scale by alpha, and add to h.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `.T` transposes; `@` is the matrix product.

```python
h = torch.zeros(2, 3); W = torch.eye(3)
torch.tanh(h) @ W.T            # zeros (2, 3)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def rnn_step(h, x, W_h, W_u, b, alpha):
    return h + alpha * (-h + torch.tanh(h) @ ____.T + x @ ____.T + b)
```

</details>

In [ ]:
# ✅ Check Task 5
H_, nin = 16, 7
hh, xx = torch.randn(4, H_, generator=g), torch.randn(4, nin, generator=g)
Wh, Wu, bb = torch.randn(H_, H_, generator=g), torch.randn(H_, nin, generator=g), torch.randn(H_, generator=g)
assert torch.allclose(rnn_step(hh, xx, Wh, Wu, bb, 0.5), ref.rnn_step(hh, xx, Wh, Wu, bb, 0.5), atol=1e-5), \
    "rnn_step differs: with rows as batch items use tanh(h) @ W_h.T and x @ W_u.T, add b, subtract h, scale by alpha, add to h"
print("✅ recurrent step works")

### Task 6 · The imitation loss (condition 1)
Training runs the whole loop (controller → muscles → body → proprioception → controller) for 2 s windows and
backpropagates through all of it, like MotorNet. The loss has four parts, the same as Lazzari & Saxena's:

| Term | Formula | Why |
|---|---|---|
| pose error | mean over valid frames of ((q − target) / 30°)² | imitate the recorded movement |
| effort | λ<sub>effort</sub> × mean(u²) | real muscles are economical; without this, antagonists fight each other |
| activity | λ<sub>rate</sub> × mean(\|tanh h\|) | low firing rates (Almani et al., regulariser 1) |
| weights | λ<sub>weight</sub> × mean \|W\| over W_h, W_u, W_o | simple connectivity (regulariser 2) |

In [ ]:
def imitation_loss(q_deg: torch.Tensor, target_deg: torch.Tensor, valid: torch.Tensor, u: torch.Tensor,
                   rates: torch.Tensor, ctrl: "ref.Controller", lam_effort: float = 0.3, lam_rate: float = 1e-3,
                   lam_weight: float = 1e-4) -> tuple[torch.Tensor, torch.Tensor]:
    """Condition-1 loss.

    q_deg, target_deg : (B, T, J) simulated and target pose (degrees)
    valid             : (B, T) 1 where the target is a real (tracked) frame
    u                 : (B, T, M) excitations;  rates : (B, T, H) tanh(h)
    ctrl              : has ctrl.W_h, ctrl.W_u, ctrl.W_o

    Returns (total, pose_mse): pose_mse = sum over frames of valid * mean_j(((q − target)/30)^2) / number of valid
    frames; total = pose_mse + lam_effort*mean(u^2) + lam_rate*mean(|rates|) + lam_weight*(sum of mean |W|).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 6? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Squared error per joint in units of 30°, averaged over joints → (B, T).
2. Multiply by `valid`, sum, divide by the number of valid frames (at least 1).
3. Add the three regularisers.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `.mean(-1)` averages the last axis; `valid.sum().clamp(min=1)` avoids dividing by 0.
- `sum(p.abs().mean() for p in (A, B, C))` adds the three mean absolute weights.

```python
err = torch.ones(2, 3, 20); valid = torch.tensor([[1., 0., 1.], [1., 1., 1.]])
(err.mean(-1) * valid).sum() / valid.sum()       # tensor(1.)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def imitation_loss(q_deg, target_deg, valid, u, rates, ctrl, lam_effort=0.3, lam_rate=1e-3, lam_weight=1e-4):
    err = ((q_deg - target_deg) / 30.0) ** 2
    mse = (err.mean(-1) * ____).sum() / valid.sum().clamp(min=1)
    l1w = sum(p.abs().mean() for p in (ctrl.W_h, ctrl.W_u, ____))
    total = mse + lam_effort * (u ** 2).mean() + lam_rate * rates.abs().mean() + ____ * l1w
    return total, mse
```

</details>

In [ ]:
# ✅ Check Task 6
ctrl_demo = ref.Controller(n_tasks, ref.N_REF, seed=0)
qs, ts = torch.randn(2, 10, ref.J, generator=g) * 20, torch.randn(2, 10, ref.J, generator=g) * 20
vs = (torch.rand(2, 10, generator=g) > 0.3).float()
us, rs = torch.rand(2, 10, ref.M, generator=g), torch.tanh(torch.randn(2, 10, 128, generator=g))
mine, theirs = imitation_loss(qs, ts, vs, us, rs, ctrl_demo), ref.imitation_loss(qs, ts, vs, us, rs, ctrl_demo, lam_effort=0.3)
assert all(torch.allclose(x, y, atol=1e-5) for x, y in zip(mine, theirs)), "imitation_loss differs"
print("✅ imitation loss works")

### Task 7 · The EMG-matching term (condition 2)
Condition 2 adds `λ_emg × mismatch` between the **simulated** envelopes (activations through the fixed electrode ring)
and the **real** envelopes of the same training frames. To stay honest with one person:
- the electrode model is **fixed**: no per-channel gains, no fitted crosstalk;
- the only free scale is **one gain per window for all 16 channels together**, solved in closed form (the real
  envelopes were scaled the same way, one number per recording, in `prep.py`):
  $s = \langle sim, real\rangle / \langle sim, sim\rangle$, then the loss is mean((s · sim − real)²);
- the band's rotation/flip (`ROT_F`) and λ (`LAM_F`) were chosen per fold on that fold's training data only (the
  cell after the next one prints how).

In [ ]:
def emg_match_loss(sim: torch.Tensor, real: torch.Tensor) -> torch.Tensor:
    """Mismatch between simulated and real envelopes, each (B, T, E).

    For every window b: s_b = sum(sim_b * real_b) / (sum(sim_b * sim_b) + 1e-8) (sums over T and E).
    Returns mean((s_b * sim_b − real_b)^2) over everything.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 7? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. For each window, the best single gain is the least-squares slope through the origin: ⟨sim, real⟩ / ⟨sim, sim⟩.
2. Keep that gain broadcastable over (T, E) (shape (B, 1, 1)).
3. Mean squared error of the scaled simulation.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `x.sum(dim=(1, 2), keepdim=True)` sums over time and channels but keeps shape (B, 1, 1).

```python
sim = torch.ones(1, 2, 2); real = 3 * sim
(sim * real).sum(dim=(1, 2), keepdim=True) / (sim * sim).sum(dim=(1, 2), keepdim=True)   # tensor([[[3.]]])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def emg_match_loss(sim, real):
    s = (sim * real).sum(dim=(1, 2), keepdim=True) / ((sim * ____).sum(dim=(1, 2), keepdim=True) + 1e-8)
    return ((s * sim - ____) ** 2).mean()
```

</details>

In [ ]:
# ✅ Check Task 7
sim_t, real_t = torch.rand(3, 20, 16, generator=g), torch.rand(3, 20, 16, generator=g)
assert torch.allclose(emg_match_loss(sim_t, real_t), ref.emg_match_loss(sim_t, real_t), atol=1e-6), \
    "emg_match_loss differs: one gain per WINDOW (sum over time and channels, keepdim), s = <sim, real> / <sim, sim>, then the mean squared error"
assert float(emg_match_loss(sim_t, 2.5 * sim_t)) < 1e-10, "a perfect match up to one gain must give 0"
print("✅ EMG-matching term works")

### Training (given code) and the checkpoints
Training uses **backpropagation through the body** (like MotorNet): 1,500 steps of Adam on batches of 128 windows of
2 s, from the **training split of the training tasks**. Every model is trained on half of the task groups
(`fold` 0 or 1, alternating groups) so that the other half are **tasks it has never seen**. Each condition is trained
with **3 seeds** × **2 folds**; condition 1 is also trained once per fold with the tendons cut (`goalc0`) for the
ablation. Training all 14 models took the time shown below on CPU cores; the notebook loads the saved checkpoints.

How the EMG-dependent choices of condition 2 were made, **separately for each fold and only from that fold's training
recordings** (so a fold's held-out tasks never influence them; the printout below shows the numbers, stored in
`checkpoints/training_summary.json`):
- **Band orientation (`ROT_F`):** fold f's goal-driven seed-0 controller simulated the **train** frames of fold f's
  training recordings; of the 32 rotations × flips, the one whose simulated envelopes correlated best with the real
  ones was kept, separately for left and right recordings.
- **λ_emg (`LAM_F`):** four values were trained per fold with the last 20% of every training recording's train split
  held out; the value with the best held-out EMG match without degrading held-out pose by more than 20% (relative to a
  goal-driven run on the same split) was kept.
- **λ_effort = 0.3** was set once from a fold-0 pilot comparing co-contraction (pose only, no EMG).

In [ ]:
TRAIN = False      # True retrains everything (about the time printed below, on a multi-core CPU)
if TRAIN:
    for cond in ("goal", "emg"):
        for seed in SEEDS:
            for fold in (0, 1):
                _, _, train_recs = ref.fold_of_groups(groups, fold)
                ctrl, norm, log = ref.train_controller(pose, pose_valid, rec, split, emg, side_of_rec, task_of_rec,
                                                       n_tasks, train_recs, cond=cond, seed=seed, iters=1500,
                                                       lam_effort=SUMMARY["lam_effort"], lam_emg=LAM_F[fold],
                                                       rot=ROT_F[fold])
                # torch.save(...) as in ref10 checkpoints if you want to keep them

models = {}
for cond in ("goal", "emg"):
    for seed in SEEDS:
        for fold in (0, 1):
            models[(cond, seed, fold)] = ref.load_controller(str(CKPT / f"{cond}_s{seed}_f{fold}.pt"))
ablation = {fold: ref.load_controller(str(CKPT / f"goalc0_s0_f{fold}.pt")) for fold in (0, 1)}
train_minutes = sum(m[2]["train_seconds"] for m in list(models.values()) + list(ablation.values())) / 60
print(f"loaded {len(models) + len(ablation)} controllers; total training time {train_minutes:.0f} CPU-process minutes")
print("Band orientation, chosen per fold on the TRAIN frames of that fold's training recordings:")
for f in (0, 1):
    for note in SUMMARY["rotation_notes"][str(f)]:
        print(f"   fold {f} {note}")
print("λ_emg, chosen per fold on a validation slice of that fold's training recordings (last 20% of each one's train split):")
for row in SUMMARY["lambda_table"]:
    print("   ", row)
print("chosen λ_emg per fold:", LAM_F, "| λ_effort =", SUMMARY["lam_effort"], "| band orientation per fold:", ROT_F)

✍️ **Q5.** Condition 2 matches the real EMG better almost by construction. Which of its results could still be *informative* (hint: what it had to change to achieve the match: effort, co-contraction, which muscles), and which result can **only** condition 1 provide?

> *Your answer:*

---
## Part 3 · Do the controllers imitate the movements?
Each controller now runs **closed loop** over whole recordings (given code). Every recording gets two runs:
- **seen task:** by the fold model that trained on its task, with its task cue. Scored on the **test** part (the last
  20% of the recording, never trained on);
- **unseen task:** by the other fold model, which never saw this task, with **no task cue**.

Seed 0 is simulated on every frame (the signatures below need the training frames too); seeds 1–2 only on the test
part, to save time.

In [ ]:
def simulate_all(cond: str, seed: int, frames: np.ndarray, bodies: dict | None = None,
                 ctrl_set: dict | None = None) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]]:
    '''(seen, unseen) runs over all recordings, assembled from the two fold models (see the text above). Each fold
    model moves its own fold's body.'''
    bodies = body_f if bodies is None else bodies
    N = len(rec)
    seen = {"q_deg": np.full((N, ref.J), np.nan), "a": np.full((N, ref.M), np.nan), "rates": np.full((N, 128), np.nan)}
    unseen = {k: v.copy() for k, v in seen.items()}
    for fold in (0, 1):
        ctrl, norm, cfg = (ctrl_set[fold] if ctrl_set is not None else models[(cond, seed, fold)])
        _, held_recs, train_recs = fold_info[fold]
        o = ref.run_segments(ctrl, bodies[fold], norm, target, rec, frames, task_of_rec, n_tasks, train_recs)
        o2 = ref.run_segments(ctrl, bodies[fold], norm, target, rec, frames, None, n_tasks, held_recs)
        for k in seen:
            m = np.isin(rec, train_recs); seen[k][m] = o[k][m]
            m = np.isin(rec, held_recs); unseen[k][m] = o2[k][m]
    return seen, unseen


import time
t0 = time.time()
SIM = {}
for cond in ("goal", "emg"):
    for seed in SEEDS:
        SIM[(cond, seed)] = simulate_all(cond, seed, np.ones(len(rec), bool) if seed == 0 else test)
SIM_ABL = simulate_all("goal", 0, np.ones(len(rec), bool), bodies=body_c0_f, ctrl_set=ablation)
print(f"simulated {len(SIM) + 1} controller pairs in {time.time() - t0:.0f} s")

In [ ]:
def block_errors(Y: np.ndarray, Y_hat: np.ndarray, mask: np.ndarray, block: np.ndarray) -> np.ndarray:
    """Mean absolute joint-angle error per block.

    Y, Y_hat : (N, J) degrees (Y_hat may contain NaN rows: frames that were not simulated)
    mask     : (N,) bool, frames to score;  block : (N,) block id of every frame

    Returns (n_blocks,) one value per block in np.unique(block) order, NaN for a block with no usable frames.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 8? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

For each block: take the frames that are in `mask`, in that block, and have no NaN in `Y_hat`. If there are any, the
block's error is the mean of |Y − Y_hat| over those frames and all joints; otherwise NaN.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.isnan(Y_hat).any(1)` → True for rows with a NaN.

```python
Y = np.zeros((4, 2)); Yh = np.array([[1, 1], [np.nan, 0], [2, 2], [3, 3.]]); b = np.array([0, 0, 1, 1])
# block 0 uses row 0 only → 1.0 ; block 1 uses rows 2, 3 → 2.5
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def block_errors(Y, Y_hat, mask, block):
    out = []
    for b in np.unique(block):
        m = mask & (block == b) & ~np.isnan(Y_hat).any(____)
        out.append(np.mean(np.abs(Y[m] - Y_hat[m])) if m.any() else ____)
    return np.array(out)
```

</details>

In [ ]:
# ✅ Check Task 8
qh = SIM[("goal", 0)][0]["q_deg"]
assert np.allclose(block_errors(pose, qh, test & pose_valid, block),
                   ref.block_errors(pose, qh, test & pose_valid, block), equal_nan=True), \
    "block_errors differs: per block, use mask frames whose Y_hat row has no NaN; NaN for a block without frames"
print("✅ block errors work")

glob_med = protocol.global_median_baseline(pose, train & pose_valid)
rec_med = protocol.per_recording_median_baseline(pose, rec, train & pose_valid)
sc = test & pose_valid
imit = {"global median": block_errors(pose, glob_med, sc, block),
        "per-recording median": block_errors(pose, rec_med, sc, block)}
for cond in ("goal", "emg"):
    for which, k in (("seen", 0), ("unseen", 1)):
        imit[f"{cond}, {which} task"] = np.nanmean([block_errors(pose, SIM[(cond, s)][k]["q_deg"], sc, block)
                                                   for s in SEEDS], axis=0)
print("Imitation error on held-out time (°), mean over 3 seeds, 95% CI over blocks:")
for name, vals in imit.items():
    mu, lo, hi = protocol.session_bootstrap_ci(vals)
    print(f"   {name:28s} {mu:5.2f}  [{lo:5.2f}, {hi:5.2f}]")
for which in ("seen", "unseen"):
    mu, lo, hi = protocol.session_bootstrap_ci(imit[f"emg, {which} task"] - imit[f"goal, {which} task"])
    print(f"   EMG-constrained minus goal-driven, {which}: {mu:+.2f} [{lo:+.2f}, {hi:+.2f}]")

# 📊 What to look for: does the simulated hand follow the recorded one on a test stretch?
r0 = int(np.unique(rec[np.isin(rec, fold_info[0][2])])[0])
m0 = (rec == r0) & test
fig, ax = plt.subplots(3, 1, figsize=(13, 6), sharex=True)
for i, j in enumerate((6, 10, 3)):
    ax[i].plot(np.where(pose_valid[m0], pose[m0, j], np.nan), "k", lw=1.5, label="recorded")
    ax[i].plot(SIM[("goal", 0)][0]["q_deg"][m0, j], label="goal-driven"); ax[i].plot(SIM[("emg", 0)][0]["q_deg"][m0, j], label="EMG-constrained")
    ax[i].set_ylabel(JOINT_NAMES[j], fontsize=7)
ax[0].legend(fontsize=7); ax[-1].set_xlabel("test frame (20 ms)"); plt.tight_layout(); plt.show()

✍️ **Q6.** The controllers get the target 100 ms ahead (as 3 eigenpostures), so imitation is not decoding. Why is error on *unseen* tasks still a meaningful number, and what does the seen − unseen gap say about what the task cue adds?

> *Your answer:*

---
## Part 4 · Does the simulated EMG look like the real EMG?
The activations go through the electrode ring (`ref.emg_by_fold`: the fixed ring with the orientation `ROT_F` of the
fold whose controller produced them,
signal-dependent noise smoothed at 10 Hz, then the same per-recording 95th-percentile scaling as `prep.py`). We compare
channel by channel with the real envelopes of the same frames, on **held-out time**, for seen and **unseen** tasks.

Because the true band orientation is unknown, you also write the alignment search used to choose `ROT_F`: try all 32
ways the 16 channels could line up and keep the best mean correlation. On *training* frames only, that is how `ROT_F`
was picked; using it on test frames would be cheating.

In [ ]:
def channel_corr(A: np.ndarray, B: np.ndarray) -> float:
    """Mean over channels of the Pearson correlation between column j of A and column j of B (both (T, E))."""
    # TODO
    raise NotImplementedError


def align_ring(sim: np.ndarray, real: np.ndarray) -> tuple[int, bool, float]:
    """Best (rotation, flip, mean channel correlation) over the 32 variants in ref.ring_variants(sim)."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 9? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `channel_corr`: centre each column, then correlation = Σ(a·b) / sqrt(Σa² · Σb²) per column; average over columns.
2. `align_ring`: loop over `ref.ring_variants(sim)` (each gives rotation, flip and the re-ordered matrix), keep the
   variant with the highest `channel_corr` against `real`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- Column-wise sums: `(A * B).sum(0)` → (E,).

```python
A = np.array([[1., 0], [2, 1], [3, 0]]); B = 2 * A
# channel_corr(A, B) = 1.0
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def channel_corr(A, B):
    A = A - A.mean(0); B = B - B.mean(0)
    return float(np.mean((A * B).sum(0) / (np.sqrt((A ** 2).sum(0) * (____ ** 2).sum(0)) + 1e-12)))


def align_ring(sim, real):
    best = (0, False, -np.inf)
    for r_, flip, Y in ref.ring_variants(sim):
        c = channel_corr(Y, ____)
        if c > best[2]:
            best = (r_, flip, c)
    return best
```

</details>

In [ ]:
# ✅ Check Task 9
A_, B_ = np.random.default_rng(0).random((200, 16)), np.random.default_rng(1).random((200, 16))
assert abs(channel_corr(A_, B_) - ref.channel_corr(A_, B_)) < 1e-9, "channel_corr differs: centre each column, then per-column Pearson, then the mean"
assert abs(channel_corr(A_, 3 * A_ + 1) - 1) < 1e-9, "a scaled, shifted copy must correlate perfectly"
assert align_ring(np.roll(A_[:, ::-1], 4, axis=1), A_)[2] > 0.999, "the search must recover a rotated, flipped copy"
print("✅ ring alignment works")

rng = np.random.default_rng(0)
emg_shift = protocol.circular_shift_null(emg, rec, rng)            # null: real EMG out of step with the movement
SEEN_BY = {f: fold_info[f][2] for f in (0, 1)}       # recordings each fold model ran with their own task cue
UNSEEN_BY = {f: fold_info[f][1] for f in (0, 1)}     # recordings each fold model never saw
SIM_EMG = {}
for (cond, seed), (seen, unseen) in SIM.items():
    SIM_EMG[(cond, seed, "seen")] = ref.emg_by_fold(seen["a"], rec, side_of_rec, SEEN_BY, ROT_F, np.random.default_rng(seed))
    SIM_EMG[(cond, seed, "unseen")] = ref.emg_by_fold(unseen["a"], rec, side_of_rec, UNSEEN_BY, ROT_F, np.random.default_rng(seed))
emg_r = {}
for cond in ("goal", "emg"):
    for which in ("seen", "unseen"):
        emg_r[(cond, which)] = np.nanmean([ref.emg_prediction_r(SIM_EMG[(cond, s, which)], emg, test, block) for s in SEEDS], 0)
        emg_r[(cond, which, "null")] = np.nanmean([ref.emg_prediction_r(SIM_EMG[(cond, s, which)], emg_shift, test, block)
                                                   for s in SEEDS], 0)
print("Simulated vs real EMG: mean channel correlation on held-out time (95% CI over blocks)")
for cond in ("goal", "emg"):
    for which in ("seen", "unseen"):
        mu, lo, hi = protocol.session_bootstrap_ci(emg_r[(cond, which)])
        dm, dl, dh = protocol.session_bootstrap_ci(emg_r[(cond, which)] - emg_r[(cond, which, "null")])
        print(f"   {cond:5s} {which:6s} r = {mu:.3f} [{lo:.3f}, {hi:.3f}]   minus shifted-EMG null {dm:+.3f} [{dl:+.3f}, {dh:+.3f}]")
mu, lo, hi = protocol.session_bootstrap_ci(emg_r[("emg", "unseen")] - emg_r[("goal", "unseen")])
print(f"   EMG-constrained minus goal-driven on UNSEEN tasks: {mu:+.3f} [{lo:+.3f}, {hi:+.3f}]")

effort = {c: np.nanmean([[np.nanmean(SIM[(c, s)][0]["a"][test & (block == b)] ** 2) for b in np.unique(block)]
                         for s in SEEDS], 0) for c in ("goal", "emg")}
cocon = {c: np.nanmean([[np.nanmean(ref.co_contraction(SIM[(c, s)][0]["a"][test & (block == b)])) for b in np.unique(block)]
                        for s in SEEDS], 0) for c in ("goal", "emg")}
for name, val in (("effort (mean a²)", effort), ("co-contraction", cocon)):
    mu, lo, hi = protocol.session_bootstrap_ci(val["emg"] - val["goal"])
    print(f"   {name:18s} goal {np.mean(val['goal']):.3f} | EMG-constrained {np.mean(val['emg']):.3f} | "
          f"difference {mu:+.3f} [{lo:+.3f}, {hi:+.3f}]")

# 📊 What to look for: a stretch of real vs simulated envelopes on an UNSEEN task
r1 = int(fold_info[0][1][0]); m1 = (rec == r1) & test
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for i, (title, X) in enumerate((("real", emg), ("goal-driven", SIM_EMG[("goal", 0, "unseen")]),
                                ("EMG-constrained", SIM_EMG[("emg", 0, "unseen")]))):
    ax[i].imshow(X[m1].T, aspect="auto", cmap="magma", vmin=0, vmax=1.5); ax[i].set_title(f"{title}: {stage_of_rec[r1]}")
ax[0].set_ylabel("electrode"); plt.tight_layout(); plt.show()

✍️ **Q7.** By how much does condition 2 beat condition 1 at predicting the real EMG of **unseen** tasks, and how does that compare with its margin over the shifted null? What did it have to pay for the better match (effort, co-contraction), and what does that suggest about how this person actually uses their muscles?

> *Your answer:*

---
## Part 5 · Does the simulation reproduce the signatures of the real EMG?
Stages 1–4b found three signatures in the real EMG. We compute the same three on simulated EMG (seed 0, seen tasks):

| Signature | Real data (earlier notebooks) | Measured here by |
|---|---|---|
| **A · few synergies** | k = 5 by the 1-point gain rule | NMF VAF on test frames for k = 1…7, k by your rule below; gap over random synergies |
| **B · small, low-variance pose subspace that is not the synergies** | rank 2–3; 9.8% of EMG variance (random 18.4%); angles to NMF ≈ random | reduced-rank regression from log-EMG (best lag) to pose: rank, variance share of the rank-3 subspace vs random, first angle to the k = 3 synergies |

**Rank** uses notebook 06's rule (`ref06.heldout_rrr_errors` over the 14 leave-one-stage-out folds, then
`ref06.choose_rank`: the smallest rank statistically indistinguishable from the best), so the stage-8 comparison matches
stage 4b; the gate uses this column. The column **(time)** shows a quicker time-split rule (smallest rank within 0.1° of
full rank on the test split) for comparison. k and the variance share use the time split, so compare **real vs
simulated within this table**.
| **C · EMG more shared across tasks than pose** | EMG sharing index ≫ pose | notebook 04's sharing index (4-D subspace per stage) |

In [ ]:
def choose_k_curve(vaf_curve: list[float], ks: list[int], min_gain: float = 1.0) -> int:
    """Smallest k after which one more synergy adds less than min_gain VAF points (the rule of notebook 02).

    vaf_curve[i] is the VAF at ks[i]. If every step gains at least min_gain, return the last k.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 10? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

Compute the gains between consecutive ks; the first gain below `min_gain` tells you to stop at the k *before* it.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.diff([70, 80, 85, 85.5])` → `[10, 5, 0.5]`: the gain from k = 3 to 4 is the first one below 1, so k = 3.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def choose_k_curve(vaf_curve, ks, min_gain=1.0):
    gains = np.diff(vaf_curve)
    for j, gain in enumerate(gains):
        if gain < ____:
            return ks[____]
    return ks[-1]
```

</details>

In [ ]:
# ✅ Check Task 10
assert choose_k_curve([70, 80, 85, 85.5], [1, 2, 3, 4]) == 3, \
    "gains are [10, 5, 0.5]: the first gain below 1 is from k = 3 to 4, so the answer is k = 3"
assert choose_k_curve([1, 3, 6], [1, 2, 3]) == 3, "if every step gains at least min_gain, return the last k"
print("✅ k rule works")

pairs_excl = ref04.simultaneous_pairs(stages, stage_of_rec, protocol.block_of_rec(filename_of_rec))


def sharing(X: np.ndarray, fit_m: np.ndarray, test_m: np.ndarray, d_sub: int = 4) -> float:
    '''Mean off-diagonal sharing index (notebook 04) of X across the 15 stages; simultaneous pairs excluded.'''
    Qd, Xd = {}, {}
    for s in stages:
        ms = task == stages.index(s)
        Qd[s] = ref04.stage_basis(X[fit_m & ms], d_sub); Xd[s] = X[test_m & ms]
    M_ = ref04.ratio_matrix(Xd, Qd); M0 = ref04.null_ratio_matrix(Xd, Qd, n_rand=10)
    return float(np.mean(ref04.off_diagonal_row_means(ref04.sharing_index(M_, M0), pairs_excl)))


def signatures(X: np.ndarray, name: str) -> dict:
    '''The three signatures of an envelope matrix X (N, 16) over all recordings (train = fit, test = score).'''
    ok = ~np.isnan(X).any(1)
    fit_m, tst = train & ok, test & ok
    ks = list(range(1, 8)); Xf, Xt = X[fit_m][::2], X[tst]
    vu, vc, Ws = [], [], {}
    for k in ks:
        _, W, _ = ref02.nmf_fit(Xf, k, n_iter=150, seed=0); Ws[k] = W
        C = ref02.nmf_transform(Xt, W, n_iter=150, seed=0)
        vu.append(ref02.vaf(Xt, C @ W)); vc.append(ref02.vaf_centred(Xt, C @ W))
    k_star = choose_k_curve(vu, ks)
    Wr = ref02.random_W(k_star, 16, 0)
    gap = vu[k_star - 1] - ref02.vaf(Xt, ref02.nmf_transform(Xt, Wr, n_iter=150, seed=0) @ Wr)
    L = ref06.log_emg(np.nan_to_num(X, nan=1.0))
    best = None
    for lag in (0, 2, 5, 10):
        F = ref06.lag_features(L, rec, [lag])
        B, mx, my, _ = ref06.rrr_fit(F[fit_m & pose_valid], pose[fit_m & pose_valid], 16)
        e = np.mean(np.abs(pose[fit_m & pose_valid] - ref06.rrr_predict(F[fit_m & pose_valid], B, mx, my)))
        best = (e, lag, F) if best is None or e < best[0] else best
    _, lag, F = best
    ranks, errs = [1, 2, 3, 4, 8, 16], []
    for rk in ranks:
        B, mx, my, _ = ref06.rrr_fit(F[fit_m & pose_valid], pose[fit_m & pose_valid], rk)
        errs.append(np.mean(np.abs(pose[tst & pose_valid] - ref06.rrr_predict(F[tst & pose_valid], B, mx, my))))
    rank_time = next(rk for rk, e in zip(ranks, errs) if e <= errs[-1] + 0.1)        # quick time-split rule
    _, ho_errs, _ = ref06.heldout_rrr_errors(F, pose, stage_of_rec, rec, filename_of_rec, pose_valid, ranks)
    rank_star = ref06.choose_rank(ho_errs, ranks)                                        # notebook 06's held-out rule
    B3, mx, _, _ = ref06.rrr_fit(F[fit_m & pose_valid], pose[fit_m & pose_valid], 3)
    Q3 = ref06.predictive_subspace(B3, 3)
    share = ref06.variance_in_subspace(F[tst], Q3)
    rng_ = np.random.default_rng(0)
    share_rand = np.mean([ref06.variance_in_subspace(F[tst], ref06.random_subspace(16, 3, rng_)) for _ in range(50)])
    angle = float(ref06.principal_angles(Q3, ref06.orthonormal(Ws[3].T))[0])
    return {"name": name, "vaf": vu, "vaf_c": vc, "k": k_star, "gap_vs_random": gap, "lag_ms": 20 * lag,
            "rank": rank_star, "rank_time": rank_time, "rank_errors": errs, "var_share": share, "var_share_random": share_rand,
            "angle_to_synergies": angle, "emg_sharing": sharing(X, fit_m, tst), "W3": Ws[3]}


t0 = time.time()
SIG = {"real": signatures(emg, "real EMG")}
POSE_SHARING = sharing(pose, train & pose_valid, test & pose_valid)
for cond in ("goal", "emg"):
    SIG[cond] = signatures(SIM_EMG[(cond, 0, "seen")], f"{cond} (seed 0)")


def show(sigs: list[dict]) -> None:
    print(f"{'':28s}{'k':>3s}{'gap vs rand':>12s}{'lag ms':>8s}{'rank':>6s}{'(time)':>7s}{'var share':>11s}{'(random)':>10s}"
          f"{'angle°':>8s}{'EMG sharing':>13s}{'pose sharing':>14s}")
    for s in sigs:
        print(f"{s['name']:28s}{s['k']:3d}{s['gap_vs_random']:12.1f}{s['lag_ms']:8d}{s['rank']:6d}{s['rank_time']:7d}{s['var_share']:11.3f}"
              f"{s['var_share_random']:10.3f}{s['angle_to_synergies']:8.0f}{s['emg_sharing']:13.2f}{POSE_SHARING:14.2f}")


show([SIG["real"], SIG["goal"], SIG["emg"]])
print(f"({time.time() - t0:.0f} s)")

# 📊 What to look for: the VAF curves, real vs simulated
plt.figure(figsize=(6, 3.5))
for key, st in (("real", "k-"), ("goal", "C0-"), ("emg", "C1-")):
    plt.plot(range(1, 8), SIG[key]["vaf"], st, marker="o", label=SIG[key]["name"])
plt.xlabel("k synergies"); plt.ylabel("test VAF (%)"); plt.legend(fontsize=7); plt.show()

✍️ **Q8.** Which of signatures A, B and C does the **goal-driven** simulation reproduce? For each one it misses, name the ingredient the simulated body or recording lacks (think: wrist muscles, co-contraction, real crosstalk, noise).

> *Your answer:*

---
## Part 6 · The key experiment: which ingredient causes each signature?
Kutch & Valero-Cuevas (2012) showed that the hand's mechanics alone can make muscle activity look like it has
"synergies". Here we take the goal-driven simulation apart in a 2 × 2:

| | crosstalk **wide** (κ × 1) | crosstalk **narrow** (κ × 4) |
|---|---|---|
| tendons **coupled** | the full model | less electrode mixing |
| tendons **cut** | separately trained controllers (`goalc0`) for a hand without shared tendons | both removed |

Crosstalk only changes the recording, so the same activations are re-recorded with a narrower pick-up. Cutting the
tendons changes the body, so those controllers were trained on the uncoupled hand (seed 0, both folds).

In [ ]:
t0 = time.time()
a_c1 = SIM[("goal", 0)][0]["a"]; a_c0 = SIM_ABL[0]["a"]
abl = {
    ("coupled", "wide"): SIG["goal"],
    ("coupled", "narrow"): signatures(ref.emg_by_fold(a_c1, rec, side_of_rec, SEEN_BY, ROT_F, np.random.default_rng(0), kappa_scale=4.0), "coupled, narrow"),
    ("cut", "wide"): signatures(ref.emg_by_fold(a_c0, rec, side_of_rec, SEEN_BY, ROT_F, np.random.default_rng(0)), "cut, wide"),
    ("cut", "narrow"): signatures(ref.emg_by_fold(a_c0, rec, side_of_rec, SEEN_BY, ROT_F, np.random.default_rng(0), kappa_scale=4.0), "cut, narrow"),
}
abl[("coupled", "wide")] = dict(abl[("coupled", "wide")], name="coupled, wide (full)")
print("Imitation with tendons cut (seen tasks, test, seed 0):",
      f"{np.nanmean(block_errors(pose, SIM_ABL[0]['q_deg'], test & pose_valid, block)):.2f}° vs coupled "
      f"{np.nanmean(block_errors(pose, SIM[('goal', 0)][0]['q_deg'], test & pose_valid, block)):.2f}°")
show([SIG["real"]] + list(abl.values()))
print(f"({time.time() - t0:.0f} s)")

# 📊 What to look for: which cell moves k, the variance share and the EMG sharing towards the real values?
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
labels = [f"{c}\n{x}" for c, x in abl]
for i, (key, title) in enumerate((("k", "synergies k"), ("var_share", "pose subspace: EMG variance share"),
                                  ("emg_sharing", "EMG sharing index"))):
    ax[i].bar(labels, [abl[k][key] for k in abl], color="C0"); ax[i].axhline(SIG["real"][key], color="k", ls="--", label="real")
    ax[i].set_title(title, fontsize=9); ax[i].tick_params(axis="x", labelsize=7)
ax[0].legend(); plt.tight_layout(); plt.show()

✍️ **Q9.** Fill in, from the table: which signature changes when you **cut the tendons**, and which when you **narrow the crosstalk**? Which real-data finding from notebooks 02–06 is therefore most likely a property of the hand or the recording rather than of the nervous system?

> *Your answer:*

---
## Part 7 · Reverse-engineering the controllers
A trained controller is a model organism we can open up completely (Almani et al. section 3.5; Sussillo & Barak
2013; Lazzari & Saxena). We analyse the **fold-0 models of all 3 seeds in both conditions**, on held-out time of the
tasks they trained on. Two rules:
- **An RNN's solution is not unique.** Only features that hold across **all 3 seeds** count (Maheswaranathan et al.
  2019); the summary at the end of this part marks which ones do.
- **Interpretations must survive causal tests:** lesions (Task 16) check whether the units and inputs we interpret
  actually matter for the movement.

### Task 11 · Slow points and their linearisation
Hold the controller's input fixed (a task cue, reference = 0, proprioception = 0: "asked to do task k at rest") and
look for hidden states where the recurrence barely moves: minimise the **speed**
$q(h) = \tfrac12\lVert F(h) - h\rVert^2$, where $F$ is one recurrent step. Near such a point the dynamics are
approximately linear, with Jacobian

$$\frac{\partial F}{\partial h} = I + \alpha\big(-I + W_h\,\mathrm{diag}(1 - \tanh^2 h)\big)$$

whose eigenvalues λ say what the controller does there: all |λ| < 1 → a **stable posture it returns to**; a complex
pair near the unit circle → **rotation**; a real λ ≈ 1 → an **integrator** (it accumulates input).

In [ ]:
def speed(ctrl: "ref.Controller", h: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """q(h) = 1/2 * ||F(h) − h||^2 for each row, with F(h) = ref.rnn_step(h, x, ctrl.W_h, ctrl.W_u, ctrl.b, ctrl.alpha).

    h : (B, H), x : (B, n_in). Returns (B,).
    """
    # TODO
    raise NotImplementedError


def jacobian(ctrl: "ref.Controller", h: np.ndarray) -> np.ndarray:
    """(H, H) Jacobian of F at h (formula above). h : (H,) numpy."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 11? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `speed`: one recurrent step, subtract h, square, sum over units, halve.
2. `jacobian`: `W_h * (1 − tanh(h)²)` scales column j of W_h by the slope of unit j; add the identity terms.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `ctrl.W_h.detach().numpy()` gives the weights as an array.
- `W * v[None, :]` multiplies column j of W by v[j].

```python
W = np.ones((2, 2)); v = np.array([1.0, 0.5])
W * v[None, :]          # [[1, 0.5], [1, 0.5]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def speed(ctrl, h, x):
    F = ref.rnn_step(h, x, ctrl.W_h, ctrl.W_u, ctrl.b, ctrl.alpha)
    return 0.5 * ((F - ____) ** 2).sum(-1)


def jacobian(ctrl, h):
    W = ctrl.W_h.detach().numpy(); H = len(h)
    return np.eye(H) + ctrl.alpha * (-np.eye(H) + W * (1.0 - np.tanh(____) ** 2)[None, :])
```

</details>

In [ ]:
# ✅ Check Task 11
c0 = models[("goal", 0, 0)][0]
hh = torch.randn(5, 128, generator=g); xx = torch.randn(5, c0.W_u.shape[1], generator=g)
assert torch.allclose(speed(c0, hh, xx), ref.speed(c0, hh, xx), atol=1e-5), \
    "speed differs: one rnn_step, minus h, squared, summed over units (last axis), times 1/2"
J_auto = torch.autograd.functional.jacobian(lambda z: ref.rnn_map(c0, z[None], xx[:1])[0], hh[0]).numpy()
assert np.allclose(jacobian(c0, hh[0].numpy()), J_auto, atol=1e-4), "Jacobian differs from autograd"
print("✅ speed and Jacobian work (the Jacobian matches automatic differentiation)")

TR0 = fold_info[0][2]                                  # recordings the fold-0 models trained on
tasks_tr = sorted(set(task_of_rec[TR0].tolist()))      # their tasks
m_re = test & np.isin(rec, TR0)                        # held-out time of trained tasks (simulated for every seed)
n_in = c0.W_u.shape[1]


def task_input(k: int) -> np.ndarray:
    x = np.zeros(n_in); x[k] = 1.0                       # task cue only; reference and proprioception at 0
    return x


t0 = time.time()
FP = {}
for cond in ("goal", "emg"):
    for seed in SEEDS:
        ctrl = models[(cond, seed, 0)][0]
        R_ = SIM[(cond, seed)][0]["rates"]
        out = {}
        for k in tasks_tr:
            pool = np.where(m_re & (task == k))[0]
            pick = np.random.default_rng(seed).choice(pool, 16, replace=False)
            starts = np.arctanh(np.clip(R_[pick], -0.999, 0.999))
            Hs, qs = ref.find_fixed_points(ctrl, task_input(k), starts, n_steps=400)
            pts = ref.unique_points(Hs, qs, q_max=1e-3, min_dist=0.5)
            slowest = Hs[np.argmin(qs)]                         # always kept: the slowest point found for task k
            eigs = [np.linalg.eigvals(jacobian(ctrl, p)) for p in (pts if len(pts) else slowest[None])]
            out[k] = {"points": pts, "slowest": slowest, "q_min": float(qs.min()), "eigs": eigs,
                      "classes": [ref.classify_eigs(e) for e in eigs]}
        FP[(cond, seed)] = out
print(f"slow-point search done in {time.time() - t0:.0f} s")
for cond in ("goal", "emg"):
    for seed in SEEDS:
        n_pts = [len(FP[(cond, seed)][k]["points"]) for k in tasks_tr]
        qmin = [FP[(cond, seed)][k]["q_min"] for k in tasks_tr]
        cls = [c for k in tasks_tr for c in FP[(cond, seed)][k]["classes"]]
        lead = [np.max(np.abs(e)) for k in tasks_tr for e in FP[(cond, seed)][k]["eigs"]]
        print(f"   {cond:5s} seed {seed}: slow points (q < 1e-3) per task {n_pts}, slowest q {min(qmin):.1e}–{max(qmin):.1e} | classes "
              f"{ {c: cls.count(c) for c in sorted(set(cls))} } | leading |λ| median {np.median(lead) if lead else np.nan:.3f}")

# 📊 What to look for: eigenvalues of each task's first slow point against the unit circle
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
th = np.linspace(0, 2 * np.pi, 200)
for i, cond in enumerate(("goal", "emg")):
    ax[i].plot(np.cos(th), np.sin(th), "k:", lw=0.8)
    for k in tasks_tr:
        if FP[(cond, 0)][k]["eigs"]:
            e = FP[(cond, 0)][k]["eigs"][0]; ax[i].scatter(e.real, e.imag, s=6, label=stages[k][:14])
    ax[i].set_aspect("equal"); ax[i].set_title(f"{cond}, seed 0: Jacobian eigenvalues"); ax[i].set_xlim(-0.2, 1.1)
ax[1].legend(fontsize=5, loc="lower left"); plt.tight_layout(); plt.show()

✍️ **Q10.** Do the controllers hold each task near a **stable** slow point, rotate, or integrate? Is the answer the same for all 3 seeds? What would an integrator-like mode mean for a hand controller (hint: what does the hand need to accumulate when it holds a posture against a spring)?

> *Your answer:*

### Task 12 · Is the controller 'really' linear around where it operates?
Reduce the hidden states to 6 principal components `z` and fit a linear dynamical system with the inputs it actually
received (the 3-D reference and the delayed joint angles): $z_{t+1} = A z_t + B x_t + c$. If a 6-D LDS predicts the
next hidden state almost perfectly, the controller behaves like a linear system near its operating points. Compare its
eigenvalues (time constants) with an LDS fitted directly to the **real pose** (as in notebook 07a).

In [ ]:
def fit_lds(Z: np.ndarray, X: np.ndarray, starts_ok: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Least squares z(t+1) = A z(t) + B x(t) + c over the steps t where starts_ok[t] is True.

    Z : (T, d) states, X : (T, m) inputs, starts_ok : (T,) bool (t and t+1 belong to the same run).
    Returns A (d, d), B (d, m), c (d,).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 12? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Rows to use: `t = np.where(starts_ok)[0]`.
2. Design matrix: `[Z[t], X[t], 1]` side by side; targets `Z[t + 1]`.
3. Solve with `np.linalg.lstsq`; the solution's rows split into A', B', c.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.hstack([Z[t], X[t], np.ones((len(t), 1))])`.
- If W solves D W ≈ Y, then `W[:d].T` is A, `W[d:d + m].T` is B, `W[-1]` is c.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def fit_lds(Z, X, starts_ok):
    t = np.where(starts_ok)[0]
    D = np.hstack([Z[t], X[t], np.ones((len(t), 1))])
    W, *_ = np.linalg.lstsq(D, Z[t + ____], rcond=None)
    d, m = Z.shape[1], X.shape[1]
    return W[:d].T, W[d:d + m].T, W[____]
```

</details>

In [ ]:
# ✅ Check Task 12
rng_l = np.random.default_rng(0)
A_true = np.array([[0.9, 0.2], [-0.2, 0.9]]); B_true = np.array([[0.5], [0.0]])
Zs, Xs = np.zeros((300, 2)), rng_l.normal(size=(300, 1))
for t_ in range(299):
    Zs[t_ + 1] = A_true @ Zs[t_] + B_true @ Xs[t_] + 0.1
A_, B_, c_ = fit_lds(Zs, Xs, np.r_[np.ones(299, bool), False])
assert np.allclose(A_, A_true, atol=1e-6), f"A not recovered (got {np.round(A_, 3)}): targets must be Z[t + 1], and A = W[:d].T"
assert np.allclose(B_, B_true, atol=1e-6), f"B not recovered (got {np.round(B_, 3)}): B = W[d:d + m].T"
assert np.allclose(c_, 0.1, atol=1e-6), f"c not recovered (got {np.round(c_, 3)}): c = the last row W[-1]"
print("✅ LDS fit recovers a known system")

ref_scores = lambda T_: ((T_ - models[("goal", 0, 0)][1].ref_mean) @ models[("goal", 0, 0)][1].ref_comps.T)
fut = np.vstack([target[5:], np.repeat(target[-1:], 5, 0)])     # target 5 frames ahead (the reference input)
same_next = np.r_[rec[1:] == rec[:-1], False]
LDS = {}
for cond in ("goal", "emg"):
    for seed in SEEDS:
        R_, Qd = SIM[(cond, seed)][0]["rates"], SIM[(cond, seed)][0]["q_deg"]
        mu_, comps_ = ref02.pca_fit(R_[m_re], 6)
        Z = np.nan_to_num(ref02.pca_transform(np.nan_to_num(R_), mu_, comps_))
        qdel = np.vstack([Qd[:2], Qd[:-2]]) - np.median(pose[train & pose_valid], 0)
        X = np.nan_to_num(np.hstack([ref_scores(fut), qdel]))
        ok = m_re & np.r_[m_re[1:], False] & same_next
        half = np.zeros(len(rec), bool)
        for r_ in TR0:
            idx = np.where(m_re & (rec == r_))[0]; half[idx[: len(idx) // 2]] = True
        A_, B_, c_ = fit_lds(Z, X, ok & half)
        tt = np.where(ok & ~half)[0]
        pred = Z[tt] @ A_.T + X[tt] @ B_.T + c_
        r2 = 1 - np.sum((Z[tt + 1] - pred) ** 2) / np.sum((Z[tt + 1] - Z[tt + 1].mean(0)) ** 2)
        LDS[(cond, seed)] = {"eig": np.linalg.eigvals(A_), "r2": r2}
mu_p, comps_p = ref02.pca_fit(pose[train & pose_valid], 6)
Zp = ref02.pca_transform(pose, mu_p, comps_p)
okp = train & pose_valid & np.r_[pose_valid[1:], False] & same_next & np.r_[train[1:], False]
A_p, _ = ref07a.fit_A(Zp[np.where(okp)[0]], Zp[np.where(okp)[0] + 1])
eig_pose = np.linalg.eigvals(A_p)
tau = lambda e: -20.0 / np.log(np.clip(np.abs(e), 1e-6, 0.999999))   # time constant in ms
print("6-D LDS of the controller's hidden state (held-out half): one-step R² and slowest time constant")
for key, v in LDS.items():
    tmax = tau(v['eig']).max()
    print(f"   {key[0]:5s} seed {key[1]}: R² {v['r2']:.3f} | slowest τ {'> 10 s' if tmax > 1e4 else f'{tmax:.0f} ms':>8s} | max |λ| {np.abs(v['eig']).max():.3f}")
print(f"   real pose (6 PCs, notebook 07a-style): slowest τ {tau(eig_pose).max():.0f} ms | max |λ| {np.abs(eig_pose).max():.3f}")

✍️ **Q11.** How well does a 6-D linear system describe the controller's hidden dynamics (R²)? Are its time constants closer to the **body** (activation 10–40 ms, passive spring ~100 ms) or to the **movement** (the real pose LDS)?

> *Your answer:*

### Task 13 · Task-vector geometry
Each task's one-hot input enters through its own column of `W_u`: the **task vector**. Lazzari & Saxena found that
similar movements get similar task vectors. Compare the cosine similarity of task vectors with how similar the tasks
really are, in pose (mean posture) and in EMG (mean muscle pattern), using the correlation of the off-diagonal entries
of the two similarity matrices (a Mantel-style test). Also compare with the distance between the tasks' slow points
from Task 11.

In [ ]:
def cosine_matrix(V: np.ndarray) -> np.ndarray:
    """(n, n) cosine similarity between the ROWS of V (n, dim)."""
    # TODO
    raise NotImplementedError


def offdiag_corr(A: np.ndarray, B: np.ndarray) -> float:
    """Pearson correlation between the off-diagonal entries of two (n, n) matrices."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 13? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `cosine_matrix`: divide each row by its length, then the matrix of dot products.
2. `offdiag_corr`: a boolean mask that is False on the diagonal; correlate `A[mask]` with `B[mask]`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.norm(V, axis=1, keepdims=True)` → row lengths (n, 1).
- `~np.eye(n, dtype=bool)` → off-diagonal mask; `np.corrcoef(a, b)[0, 1]`.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def cosine_matrix(V):
    Vn = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
    return Vn @ ____.T


def offdiag_corr(A, B):
    m = ~np.eye(len(A), dtype=bool)
    return float(np.corrcoef(A[____], B[____])[0, 1])
```

</details>

In [ ]:
# ✅ Check Task 13
Vt_ = np.random.default_rng(0).normal(size=(6, 9))
assert np.allclose(cosine_matrix(Vt_), ref.cosine_matrix(Vt_)), "cosine_matrix differs: normalise the ROWS, then Vn @ Vn.T"
assert np.allclose(np.diag(cosine_matrix(Vt_)), 1), "every row has cosine 1 with itself"
assert abs(offdiag_corr(cosine_matrix(Vt_), 2 * cosine_matrix(Vt_) + 1) - 1) < 1e-9, \
    "offdiag_corr: correlate only the off-diagonal entries (mask = ~np.eye(n, dtype=bool))"
print("✅ cosine matrix and Mantel-style correlation work")

centre = lambda M_: M_ - M_.mean(0)
pose_means = centre(np.array([pose[train & pose_valid & (task == k)].mean(0) for k in tasks_tr]))
emg_means = centre(np.array([emg[train & (task == k)].mean(0) for k in tasks_tr]))
S_pose, S_emg = cosine_matrix(pose_means), cosine_matrix(emg_means)
GEOM = {}
for cond in ("goal", "emg"):
    for seed in SEEDS:
        ctrl = models[(cond, seed, 0)][0]
        S_tv = cosine_matrix(ctrl.W_u.detach().numpy()[:, tasks_tr].T)
        fp_loc = np.array([FP[(cond, seed)][k]["slowest"] for k in tasks_tr])
        D_fp = np.sqrt(((fp_loc[:, None] - fp_loc[None]) ** 2).sum(-1))
        GEOM[(cond, seed)] = {"S_tv": S_tv, "vs_pose": offdiag_corr(S_tv, S_pose), "vs_emg": offdiag_corr(S_tv, S_emg),
                              "fp_vs_tv": offdiag_corr(-D_fp, S_tv)}
for cond in ("goal", "emg"):
    for key in ("vs_pose", "vs_emg", "fp_vs_tv"):
        vals = [GEOM[(cond, s)][key] for s in SEEDS]
        print(f"   {cond:5s} task-vector similarity {key:8s}: " + ", ".join(f"{x:+.2f}" for x in vals))

# 📊 What to look for: block structure in the task-vector similarity (seed 0)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for i, (title, S_) in enumerate((("pose similarity", S_pose), ("goal-driven task vectors", GEOM[("goal", 0)]["S_tv"]),
                                 ("EMG-constrained task vectors", GEOM[("emg", 0)]["S_tv"]))):
    ax[i].imshow(S_, cmap="RdBu_r", vmin=-1, vmax=1); ax[i].set_title(title, fontsize=9)
    ax[i].set_yticks(range(len(tasks_tr)), [stages[k][:16] for k in tasks_tr], fontsize=6); ax[i].set_xticks([])
plt.tight_layout(); plt.show()

✍️ **Q12.** Do similar tasks get similar task vectors, and is that consistent across seeds? Does the EMG-constrained controller's task geometry follow the **EMG** similarity more than the goal-driven one does? What would that mean?

> *Your answer:*

### Task 14 · Specialised or shared units?
Following Yang et al. (2019) and Lazzari & Saxena: for every hidden unit, the **task variance** is the variance of its
activity over time within each task. Normalise each unit's task variances by its largest one and cluster the units
(k-means, `ref.kmeans`). The **silhouette** score (−1 … 1) says how well separated the clusters are; pick the number of
clusters with the best silhouette. Groups of units active in only one task = specialisation; units active across tasks
= shared machinery.

In [ ]:
def task_variance(rates: np.ndarray, task: np.ndarray, tasks: list[int]) -> np.ndarray:
    """(H, n_tasks): variance over time of each unit's rate within each task. rates (T, H), task (T,)."""
    # TODO
    raise NotImplementedError


def silhouette(X: np.ndarray, lab: np.ndarray) -> float:
    """Mean silhouette of a clustering: for each point, a = mean distance to its own cluster (excluding itself),
    b = smallest mean distance to another cluster, s = (b − a) / max(a, b) (0 for a point alone in its cluster)."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 14? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `task_variance`: for each task, take the rows of `rates` from that task and their variance down the time axis;
   stack the columns.
2. `silhouette`: the full distance matrix first; then loop over points: own-cluster mean (divide by size − 1),
   other clusters' means, and the formula.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.stack([..., ...], axis=1)` turns a list of (H,) arrays into (H, n).
- `np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))` → all pairwise distances.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def task_variance(rates, task, tasks):
    return np.stack([rates[task == k].var(axis=____) for k in tasks], axis=1)


def silhouette(X, lab):
    Dm = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    s = []
    for i in range(len(X)):
        own = lab == lab[i]
        if own.sum() <= 1:
            s.append(0.0); continue
        a = Dm[i, own].sum() / (own.sum() - ____)
        b = min(Dm[i, lab == c].mean() for c in np.unique(lab) if c != lab[i])
        s.append((b - a) / max(a, b))
    return float(np.mean(s))
```

</details>

In [ ]:
# ✅ Check Task 14
Rt, tk = np.random.default_rng(0).normal(size=(300, 8)), np.repeat([0, 1, 2], 100)
assert np.allclose(task_variance(Rt, tk, [0, 1, 2]), ref.task_variance(Rt, tk, [0, 1, 2])), \
    "task_variance differs: variance down the time axis (axis 0) of each task's rows, stacked as columns (H, n_tasks)"
Xc_ = np.r_[np.random.default_rng(1).normal(0, 0.1, (20, 2)), np.random.default_rng(2).normal(3, 0.1, (20, 2))]
lab_ = np.r_[np.zeros(20, int), np.ones(20, int)]
assert abs(silhouette(Xc_, lab_) - ref.silhouette(Xc_, lab_)) < 1e-9, \
    "silhouette differs: a excludes the point itself (divide by size − 1); b is the nearest OTHER cluster's mean distance"
assert silhouette(Xc_, lab_) > 0.9, "two far-apart tight clusters should score close to 1"
print("✅ task variance and silhouette work")

UNITS = {}
for cond in ("goal", "emg"):
    for seed in SEEDS:
        R_ = SIM[(cond, seed)][0]["rates"]
        TV = task_variance(R_[m_re], task[m_re], tasks_tr)
        keep = TV.sum(1) > 1e-3
        TVn = TV[keep] / TV[keep].max(1, keepdims=True)
        scores = {k: silhouette(TVn, ref.kmeans(TVn, k, seed=0)) for k in range(2, 7)}
        k_best = max(scores, key=scores.get)
        lab = ref.kmeans(TVn, k_best, seed=0)
        specific = float(np.mean((TV[keep] / TV[keep].sum(1, keepdims=True)).max(1) > 0.5))
        UNITS[(cond, seed)] = {"TVn": TVn, "lab": lab, "k": k_best, "sil": scores[k_best], "specific": specific,
                               "active": int(keep.sum()), "keep": keep}
        print(f"   {cond:5s} seed {seed}: {keep.sum()} active units | best clusters {k_best} (silhouette "
              f"{scores[k_best]:.2f}) | units with > 50% of their variance in one task: {specific:.0%}")

# 📊 What to look for: blocks = unit groups tuned to particular tasks (seed 0)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, cond in enumerate(("goal", "emg")):
    u_ = UNITS[(cond, 0)]; order = np.argsort(u_["lab"])
    ax[i].imshow(u_["TVn"][order].T, aspect="auto", cmap="viridis")
    ax[i].set_yticks(range(len(tasks_tr)), [stages[k][:16] for k in tasks_tr], fontsize=6)
    ax[i].set_xlabel("unit (sorted by cluster)"); ax[i].set_title(f"{cond}: normalised task variance")
plt.tight_layout(); plt.show()

✍️ **Q13.** Are most units shared across tasks or specialised? Lazzari & Saxena found shared low-dimensional machinery reused across movements. Does that hold here for both conditions and all seeds?

> *Your answer:*

### Task 15 · Reading out: from hidden units to muscles
Two questions about the output side:
1. **Controller synergies.** NMF of the simulated muscle activations gives "synergies" in muscle space. Pushed
   through the electrode ring they become synergies in electrode space, which we can compare with the **real** EMG
   synergies (matched cosine after the best ring alignment, `ref03.best_match`; null = random non-negative patterns).
2. **Which muscles carry the pose-relevant latents?** Reduced-rank regression (notebook 06) from the activations
   (100 ms earlier) to pose gives a 3-D muscle-space subspace. Its biggest loadings name the muscles that matter for
   movement; in condition 2 this is a physically constrained *unmixing* of the 16 electrodes.

In [ ]:
def top_muscles(Q: np.ndarray, names: list[str], n: int = 3) -> list[list[str]]:
    """For each column of Q (M, r), the names of the n muscles with the largest |loading|, largest first."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 15? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

For each column: sort the absolute loadings in decreasing order, take the first n indices, look up their names.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.argsort(-np.abs(q))[:n]` → indices of the n largest |values|.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def top_muscles(Q, names, n=3):
    return [[names[i] for i in np.argsort(-np.abs(Q[:, j]))[:____]] for j in range(Q.shape[____])]
```

</details>

In [ ]:
# ✅ Check Task 15
Qt_ = np.zeros((4, 2)); Qt_[2, 0], Qt_[0, 0], Qt_[3, 1] = -0.9, 0.5, 0.7
tm_ = top_muscles(Qt_, list("abcd"), 2)
assert tm_[0] == ["c", "a"] and tm_[1][0] == "d", f"got {tm_}"
print("✅ top muscles works")

_, W_real4, _ = ref02.nmf_fit(emg[train][::2], 4, n_iter=150, seed=0)
G_right = ref.electrode_gains(rotation=ROT_F[0][2], flip=bool(ROT_F[0][3]))   # fold-0 models are analysed here
rng_n = np.random.default_rng(0)
null_match = np.array([ref03.best_match(W_real4, ref03.random_synergies(4, rng_n)) for _ in range(30)])
READ = {}
for cond in ("goal", "emg"):
    for seed in SEEDS:
        A_ = SIM[(cond, seed)][0]["a"]
        _, Wm, _ = ref02.nmf_fit(A_[m_re][::2], 4, n_iter=150, seed=0)               # (4, 32) muscle-space synergies
        We = (G_right @ Wm.T).T; We = We / np.linalg.norm(We, axis=1, keepdims=True)  # in electrode space
        match = ref03.best_match(W_real4, We)
        Al = ref06.lag_features(np.nan_to_num(A_), rec, [5])
        fm = m_re & pose_valid
        Bm, mxm, _, _ = ref06.rrr_fit(Al[fm], pose[fm], 3)
        Qm = ref06.predictive_subspace(Bm, 3)
        READ[(cond, seed)] = {"match": match, "Wm": Wm, "top": top_muscles(Qm, MUSCLES, 3)}
print(f"real-EMG synergies vs random patterns (null): matched cosine {null_match.mean():.2f} (95th pct {np.quantile(null_match, 0.95):.2f})")
for key, v in READ.items():
    print(f"   {key[0]:5s} seed {key[1]}: controller synergies vs real {v['match']:.2f} | muscles carrying the "
          f"pose-relevant latents: " + " / ".join(",".join(t) for t in v["top"]))

# 📊 What to look for: the muscle-space synergies (seed 0); which muscle groups work together?
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5), sharey=True)
for i, cond in enumerate(("goal", "emg")):
    ax[i].imshow(READ[(cond, 0)]["Wm"], aspect="auto", cmap="magma")
    ax[i].set_xticks(range(ref.M), MUSCLES, rotation=90, fontsize=6); ax[i].set_title(f"{cond}: controller synergies (muscles)")
plt.tight_layout(); plt.show()

✍️ **Q14.** Are the controller synergies closer to the real EMG synergies than random patterns are, and in which condition? Which lumped muscles carry the pose-relevant latents: forearm (seen by the band) or hand muscles (nearly invisible)? How could that explain why the real pose subspace holds so little EMG variance?

> *Your answer:*

### Task 16 · Lesions and input attribution: causal tests
An interpretation is only as good as a causal test of it. On a 20 s test stretch of every trained task we remove one
thing at a time and measure how much worse the imitation gets:
- **inputs** (attribution by ablation): the task cue, the reference, or proprioception;
- **units**: silence each of the two largest unit clusters from Task 14;
- **the body**: cut the tendon coupling at test time (the controller was trained on a coupled hand).

In [ ]:
def relative_increase(errors: dict[str, float], baseline: str = "intact") -> dict[str, float]:
    """Percentage change of every entry relative to errors[baseline] (the baseline itself maps to 0.0)."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 16? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

For every key: 100 × (value − baseline value) / baseline value.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

```python
{k: 100 * (v - 2.0) / 2.0 for k, v in {"a": 3.0}.items()}     # {'a': 50.0}
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def relative_increase(errors, baseline="intact"):
    b = errors[____]
    return {k: 100.0 * (v - b) / ____ for k, v in errors.items()}
```

</details>

In [ ]:
# ✅ Check Task 16
assert relative_increase({"intact": 2.0, "x": 3.0}) == {"intact": 0.0, "x": 50.0}, \
    f"expected {{'intact': 0.0, 'x': 50.0}}, got {relative_increase({'intact': 2.0, 'x': 3.0})}"
print("✅ relative increase works")

m_les = np.zeros(len(rec), bool)
for r_ in TR0:
    idx = np.where(m_re & (rec == r_))[0]; m_les[idx[:1000]] = True
LES = {}
for cond in ("goal", "emg"):
    for seed in SEEDS:
        ctrl, norm, _ = models[(cond, seed, 0)]
        u_ = UNITS[(cond, seed)]
        units_idx = np.where(u_["keep"])[0]
        sizes = np.bincount(u_["lab"]); big = np.argsort(-sizes)[:2]
        variants = {"intact": {}, "no task cue": {"in_scale": (0, 1, 1)}, "no reference": {"in_scale": (1, 0, 1)},
                    "no proprioception": {"in_scale": (1, 1, 0)}}
        for j, cl in enumerate(big):
            mask = np.ones(128); mask[units_idx[u_["lab"] == cl]] = 0.0
            variants[f"silence cluster {j + 1} ({sizes[cl]} units)"] = {"unit_mask": mask}
        errs = {}
        for name, kw in variants.items():
            o = ref.run_segments(ctrl, body_f[0], norm, target, rec, m_les, task_of_rec, n_tasks, TR0, **kw)
            errs[name] = np.nanmean(block_errors(pose, o["q_deg"], m_les & pose_valid, block))
        o = ref.run_segments(ctrl, body_c0_f[0], norm, target, rec, m_les, task_of_rec, n_tasks, TR0)
        errs["tendons cut at test"] = np.nanmean(block_errors(pose, o["q_deg"], m_les & pose_valid, block))
        LES[(cond, seed)] = (errs, relative_increase(errs))
for cond in ("goal", "emg"):
    names = [k for k in LES[(cond, 0)][1] if not k.startswith("silence")] + ["silence cluster 1", "silence cluster 2"]
    print(f"{cond}: % increase in imitation error (seeds 0, 1, 2)")
    for nm in names:
        vals = [next((v for k, v in LES[(cond, s)][1].items() if k.startswith(nm)), np.nan) for s in SEEDS]
        print(f"   {nm:22s} " + "  ".join(f"{x:+7.0f}%" for x in vals))

✍️ **Q15.** Rank the inputs by how much the controllers depend on them. Is the ranking the same across seeds and conditions? Does silencing a unit cluster hurt as much as its size suggests? What does the 'tendons cut at test' lesion say about how much the controller has learned the body's coupling?

> *Your answer:*

### Task 17 · Robustness across seeds and conditions
Different seeds can learn different solutions. Compare the hidden activity of all 6 controllers on the same test
frames with two measures: **CCA** (mean of the top 10 canonical correlations: shared linear information) and
**Procrustes disparity** on the top 10 principal components (shape difference after the best rotation and scaling;
0 = same geometry, 1 = unrelated).

In [ ]:
def procrustes_disparity(X: np.ndarray, Y: np.ndarray) -> float:
    """1 − (sum of singular values of X'Y)^2 after centring each matrix and scaling it to unit Frobenius norm.
    X, Y : (T, d), the same time points."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 17? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Centre both (subtract column means) and divide each by its Frobenius norm.
2. The best rotation's overlap is the sum of the singular values of X'Y; disparity = 1 − (that sum)².

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.norm(X)` is the Frobenius norm; `np.linalg.svd(M, compute_uv=False)` gives singular values only.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def procrustes_disparity(X, Y):
    X = X - X.mean(0); Y = Y - Y.mean(0)
    X = X / np.linalg.norm(X); Y = Y / np.linalg.norm(____)
    s = np.linalg.svd(X.T @ Y, compute_uv=False)
    return float(1.0 - s.____() ** 2)
```

</details>

In [ ]:
# ✅ Check Task 17
Xp = np.random.default_rng(0).normal(size=(200, 5)); Rq, _ = np.linalg.qr(np.random.default_rng(1).normal(size=(5, 5)))
assert abs(procrustes_disparity(Xp, 3 * Xp @ Rq + 1)) < 1e-9, "a rotated, scaled copy must have disparity 0"
assert abs(procrustes_disparity(Xp, Xp[::-1]) - ref.procrustes_disparity(Xp, Xp[::-1])) < 1e-9, \
    "procrustes_disparity differs: centre, scale to unit Frobenius norm, 1 − (sum of singular values of X'Y)^2"
print("✅ Procrustes disparity works")

keys = [(c, s) for c in ("goal", "emg") for s in SEEDS]
ok_rows = m_re & ~np.isnan(np.c_[[SIM[k][0]["rates"][:, 0] for k in keys]].T).any(1)
PCS = {}
for k in keys:
    R_ = SIM[k][0]["rates"][ok_rows]; mu_, cp_ = ref02.pca_fit(R_, 10); PCS[k] = ref02.pca_transform(R_, mu_, cp_)
CCA_M = np.array([[ref.cca_corrs(PCS[a], PCS[b]).mean() for b in keys] for a in keys])
PRO_M = np.array([[procrustes_disparity(PCS[a], PCS[b]) for b in keys] for a in keys])
same = np.array([[a[0] == b[0] and a != b for b in keys] for a in keys])
diff = np.array([[a[0] != b[0] for b in keys] for a in keys])
print(f"CCA (top-10 mean):   within condition {CCA_M[same].mean():.2f} | across conditions {CCA_M[diff].mean():.2f}")
print(f"Procrustes disparity: within condition {PRO_M[same].mean():.2f} | across conditions {PRO_M[diff].mean():.2f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for i, (title, Mx) in enumerate((("CCA", CCA_M), ("Procrustes disparity", PRO_M))):
    im = ax[i].imshow(Mx, cmap="viridis"); ax[i].set_title(title)
    ax[i].set_xticks(range(6), [f"{c[:4]} s{s}" for c, s in keys], fontsize=7, rotation=45)
    ax[i].set_yticks(range(6), [f"{c[:4]} s{s}" for c, s in keys], fontsize=7); plt.colorbar(im, ax=ax[i])
plt.tight_layout(); plt.show()

### What holds across seeds?
The table below collects the interpretation results per seed. A feature is **robust** if it has the same answer in all
3 seeds (categorical) or the same sign (numbers). Only robust features go into the gate.

In [ ]:
def robust(vals: list) -> str:
    if all(isinstance(v, str) for v in vals):
        return "robust" if len(set(vals)) == 1 else "varies"
    v = np.array(vals, float)
    if not (np.all(v > 0) or np.all(v < 0)):
        return "varies"
    return "robust" if np.min(np.abs(v)) >= 0.1 else "robust sign, weak (some |value| < 0.1)"


rows = []
for cond in ("goal", "emg"):
    feats = {
        "dominant slow-point class": [max(set(c := [x for k in tasks_tr for x in FP[(cond, s)][k]["classes"]] or ["none"]), key=c.count)
                                      for s in SEEDS],
        "6-D LDS describes the hidden state (R² > 0.95)": ["yes" if LDS[(cond, s)]["r2"] > 0.95 else "no" for s in SEEDS],
        "similar task vectors ~ nearby slow points": [GEOM[(cond, s)]["fp_vs_tv"] for s in SEEDS],
        "task vectors ~ pose similarity": [GEOM[(cond, s)]["vs_pose"] for s in SEEDS],
        "task vectors ~ EMG similarity": [GEOM[(cond, s)]["vs_emg"] for s in SEEDS],
        "share of task-specific units − 0.5": [UNITS[(cond, s)]["specific"] - 0.5 for s in SEEDS],
        "top muscle of latent 1": [READ[(cond, s)]["top"][0][0] for s in SEEDS],
        "most important input": [max(("no task cue", "no reference", "no proprioception"), key=lambda n: LES[(cond, s)][1][n])
                                 for s in SEEDS],
        "controller synergies − null 95th pct": [READ[(cond, s)]["match"] - np.quantile(null_match, 0.95) for s in SEEDS],
    }
    for name, vals in feats.items():
        shown = ", ".join(v if isinstance(v, str) else f"{v:+.2f}" for v in vals)
        rows.append((cond, name, shown, robust(vals)))
for cond, name, shown, verdict in rows:
    print(f"   {cond:5s} {name:46s} {shown:40s} {verdict}")

✍️ **Q16.** Which interpretation results are robust across seeds, and which differ between the goal-driven and EMG-constrained controllers? In one or two sentences: what changes inside a controller when it must also match this person's EMG?

> *Your answer:*

---
## Part 8 · Compositional transfer: learn a new task by fitting only its task vector
Lazzari & Saxena's test, as in notebook 07b, now on the simulated controllers: for every task group a fold model
**never saw**, use the first `s` seconds of each of its recordings to calibrate, and score imitation on the second half.

| Method | What is fitted |
|---|---|
| none | nothing: no task cue |
| nearest | nothing: reuse the trained task whose mean posture is closest to the calibration frames |
| task vector only | a new 128-number input vector, everything else frozen |
| all weights | every weight of the controller (the usual "recalibrate everything") |

Fitting runs gradient descent through the body, so the full sweep (2 conditions × 14 groups × 4 budgets × 2 fits)
was run once and saved; below you write the windowing step and run one live example.

In [ ]:
def calibration_windows(cal_mask: np.ndarray, rec: np.ndarray, T: int, stride: int = 10) -> np.ndarray:
    """First frames of every T-frame window that lies entirely inside cal_mask and inside one recording, stepping by
    `stride` inside every contiguous run of cal_mask frames. Returns an int array of frame indices."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 18? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. For each recording that has calibration frames, get their indices.
2. Split them into contiguous runs (where the index jumps by more than 1).
3. In each run, every `stride`-th position that still leaves T frames is a window start.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)` → list of contiguous runs.
- `range(0, len(run) - T + 1, stride)` → valid offsets.

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def calibration_windows(cal_mask, rec, T, stride=10):
    starts = []
    for r_ in np.unique(rec[cal_mask]):
        idx = np.where((rec == r_) & cal_mask)[0]
        for run in np.split(idx, np.where(np.diff(idx) != 1)[0] + ____):
            for s0 in range(0, len(run) - T + 1, ____):
                starts.append(run[s0])
    return np.array(starts, dtype=np.int64)
```

</details>

In [ ]:
# ✅ Check Task 18
cm_ = np.zeros(len(rec), bool); cm_[100:300] = True; cm_[500:520] = True
assert np.array_equal(calibration_windows(cm_, rec, 50), ref.calibration_windows(cm_, rec, 50)), \
    f"calibration windows differ: expected {ref.calibration_windows(cm_, rec, 50)[:6]}..., got {calibration_windows(cm_, rec, 50)[:6]}..."
print("✅ calibration windows work")

# one live example: the first held-out group of fold 0, 5 s of calibration, task vector only (goal-driven, seed 0)
name0, recs0 = [g_ for i, g_ in enumerate(groups) if i % 2 == 0][0]
ctrl, norm, _ = models[("goal", 0, 0)]
(_, cal5, ev5), = protocol.calibration_splits(np.isin(rec, recs0), rec, [5])
t0 = time.time()
_, vec = ref.fit_task_vector(ctrl, body_f[0], norm, target, pose_valid, calibration_windows(cal5, rec, 50), steps=15)
e_none = ref.run_segments(ctrl, body_f[0], norm, target, rec, ev5, None, n_tasks, recs0)["q_deg"]
e_vec = ref.run_segments(ctrl, body_f[0], norm, target, rec, ev5, None, n_tasks, recs0, task_vec=vec)["q_deg"]
mm = ev5 & pose_valid
print(f"{name0}: no task cue {np.nanmean(np.abs(pose[mm] - e_none[mm])):.2f}° → fitted task vector "
      f"{np.nanmean(np.abs(pose[mm] - e_vec[mm])):.2f}° (15 steps, {time.time() - t0:.0f} s)")

TR = np.load(CKPT / "transfer_results.npz", allow_pickle=True)
budgets = TR["budgets"]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for i, cond in enumerate(("goal", "emg")):
    print(f"{cond}: imitation error on the evaluation half (°), mean and 95% CI over the 14 held-out groups")
    for meth, col in (("none", "gray"), ("nearest", "C2"), ("vector", "C0"), ("all", "C3")):
        E = TR[f"{cond}_{meth}"]                                   # (14 groups, budgets)
        ci = [protocol.session_bootstrap_ci(E[:, b]) for b in range(len(budgets))]
        mu = [c[0] for c in ci]
        ax[i].plot(budgets, mu, "o-", color=col, label=meth)
        ax[i].fill_between(budgets, [c[1] for c in ci], [c[2] for c in ci], color=col, alpha=0.15)
        print(f"   {meth:8s} " + "  ".join(f"{b:>2d}s {m:5.2f}" for b, m in zip(budgets, mu)))
    for b_i in range(1, len(budgets)):
        d_ = protocol.session_bootstrap_ci(TR[f"{cond}_vector"][:, b_i] - TR[f"{cond}_all"][:, b_i])
        print(f"   vector − all at {budgets[b_i]:>2d} s: {d_[0]:+.2f} [{d_[1]:+.2f}, {d_[2]:+.2f}]")
    ax[i].set_title(cond); ax[i].set_xlabel("seconds of the new task")
ax[0].set_ylabel("imitation error (°)"); ax[0].legend(); plt.tight_layout(); plt.show()

✍️ **Q17.** Does fitting only the task vector beat recalibrating all weights with little data, as in notebook 07b's real-data RNN? The simulated controllers receive the target as an input: why might that make the task cue, and hence task-vector transfer, matter **less** here than in 07b?

> *Your answer:*

---
## Part 9 · Virtual people: hypotheses for stage 7
Keep the controller and its muscle activity fixed, and change only how the band sits and records: a different
rotation or mirroring, wider or narrower crosstalk, and per-electrode gain differences (skin contact). How much do the
synergies and the pose subspace change? These are **hypotheses for the multi-user dataset**, not evidence: real people
also differ in anatomy and in how they move.

In [ ]:
def virtual_person_gains(rotation: int, flip: bool, kappa_scale: float, gain_sd: float, rng: np.random.Generator
                         ) -> np.ndarray:
    """(16, M) electrode mixing for a virtual person: ref.electrode_gains(kappa_scale=..., rotation=..., flip=...)
    with every electrode's row multiplied by its own log-normal gain exp(N(0, gain_sd))."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 19? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

Build the ring with `ref.electrode_gains`, draw 16 gains `np.exp(rng.normal(0, gain_sd, 16))`, and scale the rows.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `G * g[:, None]` multiplies row e of G by g[e].

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def virtual_person_gains(rotation, flip, kappa_scale, gain_sd, rng):
    G = ref.electrode_gains(kappa_scale=kappa_scale, rotation=rotation, flip=flip)
    return G * np.exp(rng.normal(0, ____, 16))[:, ____]
```

</details>

In [ ]:
# ✅ Check Task 19
Gv = virtual_person_gains(3, True, 2.0, 0.0, np.random.default_rng(0))
assert np.allclose(Gv, ref.electrode_gains(kappa_scale=2.0, rotation=3, flip=True)), "with gain_sd = 0 nothing is jittered"
assert virtual_person_gains(0, False, 1.0, 0.3, np.random.default_rng(0)).shape == (16, ref.M), "the result must be (16 electrodes, M muscles)"
print("✅ virtual person gains work")

right = (np.isin(rec, np.where(side_of_rec == "right")[0]) & np.isin(rec, fold_info[0][2])
         & ~np.isnan(SIM[("goal", 0)][0]["a"]).any(1))                 # right-hand recordings simulated by fold 0
A_r = SIM[("goal", 0)][0]["a"][right]; rec_r = rec[right]; tr_r = train[right]
G_ref = ref.electrode_gains(rotation=ROT_F[0][2], flip=bool(ROT_F[0][3]))
X_ref = ref.simulate_emg(A_r, G_ref, rec_r, np.random.default_rng(0))
_, W_ref, _ = ref02.nmf_fit(X_ref[tr_r][::2], 4, n_iter=150, seed=0)
rng_v = np.random.default_rng(1)
VP = []
for i in range(8):
    kap = [0.5, 1.0, 2.0][i % 3]
    Gp = virtual_person_gains(int(rng_v.integers(16)), bool(rng_v.integers(2)), kap, 0.3, rng_v)
    Xp = ref.simulate_emg(A_r, Gp, rec_r, np.random.default_rng(i))
    _, Wp, _ = ref02.nmf_fit(Xp[tr_r][::2], 4, n_iter=150, seed=0)
    raw = float(ref03.match_synergies(W_ref, Wp)[0].mean())
    aligned = ref03.best_match(W_ref, Wp)
    VP.append((kap, raw, aligned))
    print(f"   virtual person {i}: kappa x{kap:3.1f} | synergy similarity to the reference person: raw {raw:.2f}, "
          f"after ring alignment {aligned:.2f}")
print(f"mean raw {np.mean([v[1] for v in VP]):.2f} vs aligned {np.mean([v[2] for v in VP]):.2f} "
      f"(random synergies ≈ {null_match.mean():.2f} against the real ones)")

✍️ **Q18.** If only the band moves, how similar do the same person's synergies look before and after aligning the ring? What does that predict for comparing synergies **across** people in stage 7, and which part of the difference between real people could this simulation never capture?

> *Your answer:*

---
## ⚠️ Methodology: what this notebook can and can't tell you
Read this before writing the Result. Each row is a reason the numbers could look better (or different) than the truth, and what would test it.

| Issue | Why it matters | What would test it |
|---|---|---|
| **One person** | Every number describes one participant's 15 recording blocks; the simulation adds structure but no new people | Repeat on a multi-user subset (stage 7) |
| **Literature parameters, not fitted** | Moment arms, strengths, muscle positions and crosstalk are rough textbook values, fixed in advance; a different but equally plausible set could change the signatures | Sweep the parameters (Part 9 does this for the electrodes) and report which conclusions survive |
| **Lumped, simplified muscles** | 32 force generators with constant moment arms, no force-length-velocity, no tendon slack, no wrist | A full musculoskeletal hand (MyoSuite) as a realism check |
| **No wrist muscles** | The real band sits over large wrist flexors/extensors whose activity is unrelated to finger pose; the simulation has none, so it lacks a big source of 'private' EMG variance | Add wrist muscles driven by the WristFlexionAbduction task, or estimate how much real EMG variance they could carry |
| **Imitation is not control** | The controller is given the target 100 ms ahead (as 3 eigenpostures); a real nervous system plans the movement itself | Train with goals only (end postures) as in Lazzari & Saxena |
| **Condition 2 is not a test of emergence** | It was trained on the EMG, so matching it proves nothing; only what it had to change (effort, co-contraction, muscles) is informative | Use condition 1 for every 'does the body explain it' claim |
| **Zero-phase envelopes** | The real envelopes contain ~100 ms of future EMG; the simulated activations are causal | Compare with causally filtered real EMG |
| **Seeds and folds** | 3 seeds × 2 folds; the transfer sweep uses seed 0; RNN solutions vary | Report only seed-robust features (Part 7 table); more seeds if a feature is borderline |
| **Simulation ≠ nervous system** | A controller that reproduces the EMG shows one sufficient explanation, not the real one | Look for predictions that differ between explanations and test them on data |

---
## Result · gate for stage 8
**Question:** Does a goal-driven simulated hand reproduce the structure of the real EMG, which ingredient causes each signature, and what do the controllers learn (robustly across seeds)?

| Measure | Value (mean, 95% CI over sessions) | Baseline / null | Pass? |
|---|---|---|---|
| Imitation error, seen / unseen tasks (goal, emg) | | | |
| Sim–real EMG correlation on unseen tasks vs shifted null (goal, emg) | | | |
| Effort and co-contraction, emg − goal | | | |
| Signature A (k) reproduced? (goal) | | | |
| Signature B (pose subspace: rank, variance share, angle) reproduced? (goal) | | | |
| Signature C (EMG > pose sharing) reproduced? (goal) | | | |
| Ablation: ingredient needed for A / B / C | | | |
| Robust interpretation features (Part 7 table) | | | |
| Task-vector transfer vs all weights at 2–5 s | | | |

**Gate:** the goal-driven simulation reproduces ≥ 2 of the 3 signatures, the ablation identifies which ingredient each one needs, and the interpretation results used are robust across all 3 seeds.

- **Pass →** treat those signatures in notebooks 02–06 as properties of the hand and the recording, design stage 7 to align bands across people, and use the seed-robust controller features as hypotheses for the multi-user data.
- **Fail →** the simulation as built cannot explain the real EMG: name the missing ingredient (Methodology table) and add it before drawing conclusions about the nervous system.
- **Was your prediction right? Why or why not?**